# SpaMGCL Formal Benchmark: 5 Datasets × 10 Seeds × 400 Epochs

This notebook performs the final repeated benchmark experiments for SpaMGCL.

Protocol:
- Datasets: HLN-A1, HLN-D1, E18.5, S2-E15, S2-E18
- Training epochs: 400
- Training seeds: 0–9
- Warm-up: 10 epochs
- Official representation: concat-Z
- Refinement: BSRR
- BSRR spatial_k: 3
- Clustering: KMeans
- KMeans n_init: 20
- KMeans random_state: 0
- NMI average method: max

All datasets use one unified training budget.
Final performance will be reported as mean ± standard deviation over 10 independent training seeds.

Cell 1：检查 Kaggle 环境

In [1]:
# ============================================================
# Cell 1
# Environment check
# ============================================================

import sys
import torch
import numpy as np
import sklearn
import yaml


print("=" * 80)
print("KAGGLE ENVIRONMENT")
print("=" * 80)

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())


if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 GPU，请先在 Kaggle Settings 中开启 GPU。"
    )


print("GPU:", torch.cuda.get_device_name(0))

print("\nPASS: Kaggle environment ready.")

KAGGLE ENVIRONMENT
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Python executable: /usr/bin/python3
PyTorch: 2.10.0+cu128
NumPy: 2.0.2
scikit-learn: 1.6.1
CUDA available: True
GPU: Tesla T4

PASS: Kaggle environment ready.


Cell 2：安装并检查 anndata

In [2]:
# ============================================================
# Cell 2
# anndata
# ============================================================

import sys
import subprocess
import importlib.util


if importlib.util.find_spec("anndata") is None:

    print("Installing anndata...")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            "anndata==0.11.4",
        ],
        check=True,
    )


import anndata


print("anndata:", anndata.__version__)
print("NumPy:", __import__("numpy").__version__)

print("\nPASS: anndata ready.")

Installing anndata...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 1.4 MB/s eta 0:00:00
anndata: 0.11.4
NumPy: 2.0.2

PASS: anndata ready.


Cell 3：重新克隆正式 SpaMGCL 代码

In [3]:
# ============================================================
# Cell 3
# Clone clean SpaMGCL repository
# ============================================================

from pathlib import Path
import shutil
import subprocess


REPO_ROOT = Path(
    "/kaggle/working/SpaMGCL"
)

PROJECT_ROOT = (
    REPO_ROOT
    / "SpaMGCL"
)


if REPO_ROOT.exists():

    print(
        "Removing existing repository:"
    )

    print(
        REPO_ROOT
    )

    shutil.rmtree(
        REPO_ROOT
    )


subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        "main",
        "https://github.com/huqian122/SpaMGCL.git",
        str(REPO_ROOT),
    ],
    check=True,
)


assert PROJECT_ROOT.exists(), (
    f"Project root not found: {PROJECT_ROOT}"
)


print("\nProject root:")
print(PROJECT_ROOT)


print("\nRepository commit:")

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_ROOT),
        "log",
        "-1",
        "--oneline",
    ],
    check=True,
)


print(
    "\nPASS: clean repository cloned."
)

Cloning into '/kaggle/working/SpaMGCL'...



Project root:
/kaggle/working/SpaMGCL/SpaMGCL

Repository commit:
b4abd3d Add files via upload

PASS: clean repository cloned.


Cell 4：检查正式代码和 5 个 clean base configs

In [4]:
# ============================================================
# Cell 4
# Check source code and final clean base configs
# ============================================================

from pathlib import Path
import os
import yaml
import subprocess
import sys


PROJECT_ROOT = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL"
)

os.chdir(
    PROJECT_ROOT
)


# ============================================================
# Base configs
# ============================================================

BASE_CONFIGS = {

    "hlna1":
        PROJECT_ROOT
        / "configs/final_clean/hlna1_clean_200.yaml",

    "d1":
        PROJECT_ROOT
        / "configs/final_clean/d1_clean_200.yaml",

    "e185":
        PROJECT_ROOT
        / "configs/final_clean/e185_clean_200.yaml",

    "s2e15":
        PROJECT_ROOT
        / "configs/final_clean/s2e15_clean_200.yaml",

    "s2e18":
        PROJECT_ROOT
        / "configs/final_clean/s2e18_clean_200.yaml",
}


EXPECTED_DATASETS = {

    "hlna1": "HLN-A1",
    "d1": "HLN-D1",
    "e185": "E18.5",
    "s2e15": "S2-E15",
    "s2e18": "S2-E18",
}


EXPECTED_CLUSTERS = {

    "hlna1": 10,
    "d1": 11,
    "e185": 14,
    "s2e15": 15,
    "s2e18": 16,
}


# ============================================================
# Required source code
# ============================================================

REQUIRED_CODE = [

    PROJECT_ROOT
    / "src/clustering/refinement.py",

    PROJECT_ROOT
    / "src/clustering/predict.py",

    PROJECT_ROOT
    / "experiments/run_exp.py",

    PROJECT_ROOT
    / "scripts/audit_run.py",

    PROJECT_ROOT
    / "tests/test_bsrr.py",
]


print("=" * 90)
print("SOURCE CODE CHECK")
print("=" * 90)


for path in REQUIRED_CODE:

    assert path.exists(), (
        f"Missing source file: {path}"
    )

    print(
        "PASS:",
        path.relative_to(
            PROJECT_ROOT
        )
    )


# ============================================================
# Python syntax check
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(
            PROJECT_ROOT
            / "experiments/run_exp.py"
        ),
        str(
            PROJECT_ROOT
            / "src/clustering/refinement.py"
        ),
        str(
            PROJECT_ROOT
            / "src/clustering/predict.py"
        ),
        str(
            PROJECT_ROOT
            / "scripts/audit_run.py"
        ),
    ],
    check=True,
)


print(
    "\nPASS: Python syntax check."
)


# ============================================================
# Config audit
# ============================================================

print("\n" + "=" * 90)
print("FINAL CLEAN BASE CONFIG AUDIT")
print("=" * 90)


for key, config_path in (
    BASE_CONFIGS.items()
):

    assert config_path.exists(), (
        f"Missing base config: "
        f"{config_path}"
    )

    with config_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        cfg = yaml.safe_load(f)


    dataset = (
        cfg["experiment"][
            "dataset"
        ]
    )

    warm_up = int(
        cfg["training"][
            "warm_up_epochs"
        ]
    )

    n_clusters = int(
        cfg["clustering"][
            "n_clusters"
        ]
    )


    print(
        f"{key:7s} | "
        f"dataset={dataset:7s} | "
        f"warmup={warm_up:2d} | "
        f"K={n_clusters:2d} | "
        f"embedding="
        f"{cfg['clustering']['embedding']} | "
        f"n_init="
        f"{cfg['clustering']['n_init']} | "
        f"kmeans_rs="
        f"{cfg['clustering']['random_state']} | "
        f"BSRR="
        f"{cfg['refinement']['enabled']} | "
        f"spatial_k="
        f"{cfg['refinement']['spatial_k']}"
    )


    assert (
        dataset
        == EXPECTED_DATASETS[key]
    )

    assert (
        warm_up
        == 10
    )

    assert (
        cfg["clustering"][
            "method"
        ].lower()
        == "kmeans"
    )

    assert (
        cfg["clustering"][
            "embedding"
        ].lower()
        == "concat_z"
    )

    assert (
        n_clusters
        == EXPECTED_CLUSTERS[key]
    )

    assert (
        int(
            cfg["clustering"][
                "n_init"
            ]
        )
        == 20
    )

    assert (
        int(
            cfg["clustering"][
                "random_state"
            ]
        )
        == 0
    )

    assert (
        cfg["refinement"][
            "enabled"
        ]
        is True
    )

    assert (
        cfg["refinement"][
            "method"
        ].lower()
        == "bsrr"
    )

    assert (
        int(
            cfg["refinement"][
                "spatial_k"
            ]
        )
        == 3
    )

    assert (
        cfg["evaluation"][
            "nmi_average_method"
        ]
        == "max"
    )


print(
    "\nPASS: all five final clean "
    "base configs verified."
)

SOURCE CODE CHECK
PASS: src/clustering/refinement.py
PASS: src/clustering/predict.py
PASS: experiments/run_exp.py
PASS: scripts/audit_run.py
PASS: tests/test_bsrr.py

PASS: Python syntax check.

FINAL CLEAN BASE CONFIG AUDIT
hlna1   | dataset=HLN-A1  | warmup=10 | K=10 | embedding=concat_z | n_init=20 | kmeans_rs=0 | BSRR=True | spatial_k=3
d1      | dataset=HLN-D1  | warmup=10 | K=11 | embedding=concat_z | n_init=20 | kmeans_rs=0 | BSRR=True | spatial_k=3
e185    | dataset=E18.5   | warmup=10 | K=14 | embedding=concat_z | n_init=20 | kmeans_rs=0 | BSRR=True | spatial_k=3
s2e15   | dataset=S2-E15  | warmup=10 | K=15 | embedding=concat_z | n_init=20 | kmeans_rs=0 | BSRR=True | spatial_k=3
s2e18   | dataset=S2-E18  | warmup=10 | K=16 | embedding=concat_z | n_init=20 | kmeans_rs=0 | BSRR=True | spatial_k=3

PASS: all five final clean base configs verified.


Cell 5：检查 Kaggle 数据集

In [5]:
# ============================================================
# Cell 5
# Dataset mount check
# ============================================================

from pathlib import Path


DATA_ROOT = Path(
    "/kaggle/input/datasets/wuvdji/smgc-data"
)


print("=" * 80)
print("DATASET CHECK")
print("=" * 80)

print(
    "Data root:",
    DATA_ROOT,
)

print(
    "Exists:",
    DATA_ROOT.exists(),
)


assert DATA_ROOT.exists(), (
    "没有找到 smgc-data。\n"
    "请先在 Kaggle Notebook 中 Add Input。"
)


print(
    "\nTop-level contents:"
)


for p in sorted(
    DATA_ROOT.iterdir()
):

    print(
        " -",
        p.name,
    )


expected_dirs = [

    DATA_ROOT
    / "Human_Lymph_Nodes",

    DATA_ROOT
    / "E18.5_mouse_brain",

    DATA_ROOT
    / "Mouse_Embryos_S2",
]


for path in expected_dirs:

    assert path.exists(), (
        f"Missing dataset directory: "
        f"{path}"
    )


print(
    "\nPASS: all real datasets mounted."
)

DATASET CHECK
Data root: /kaggle/input/datasets/wuvdji/smgc-data
Exists: True

Top-level contents:
 - E18.5_mouse_brain
 - Human_Lymph_Nodes
 - Mouse_Embryos_S2
 - simulation

PASS: all real datasets mounted.


Cell 6：生成 5 datasets × 10 seeds × 400 epochs 的正式配置

In [6]:
# ============================================================
# Cell 6
# Generate 50 formal 400-epoch configs
# ============================================================

from pathlib import Path
import copy
import yaml


PROJECT_ROOT = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/wuvdji/smgc-data"
)


FINAL_CONFIG_DIR = (
    PROJECT_ROOT
    / "configs"
    / "formal_400ep"
)

FINAL_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FINAL_RESULT_ROOT_NAME = (
    "results_final_400"
)


BASE_CONFIGS = {

    "hlna1":
        PROJECT_ROOT
        / "configs/final_clean/hlna1_clean_200.yaml",

    "d1":
        PROJECT_ROOT
        / "configs/final_clean/d1_clean_200.yaml",

    "e185":
        PROJECT_ROOT
        / "configs/final_clean/e185_clean_200.yaml",

    "s2e15":
        PROJECT_ROOT
        / "configs/final_clean/s2e15_clean_200.yaml",

    "s2e18":
        PROJECT_ROOT
        / "configs/final_clean/s2e18_clean_200.yaml",
}


EXPECTED_DATASETS = {

    "hlna1": "HLN-A1",
    "d1": "HLN-D1",
    "e185": "E18.5",
    "s2e15": "S2-E15",
    "s2e18": "S2-E18",
}


EXPECTED_CLUSTERS = {

    "hlna1": 10,
    "d1": 11,
    "e185": 14,
    "s2e15": 15,
    "s2e18": 16,
}


FINAL_CONFIGS = []


for dataset_key, base_path in (
    BASE_CONFIGS.items()
):

    with base_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        base_cfg = yaml.safe_load(f)


    for seed in range(10):

        cfg = copy.deepcopy(
            base_cfg
        )


        # ====================================================
        # Experiment
        # ====================================================

        experiment_name = (
            f"{dataset_key}_"
            f"formal_400ep_"
            f"seed{seed}"
        )


        cfg[
            "experiment"
        ]["name"] = (
            experiment_name
        )

        cfg[
            "experiment"
        ]["dataset"] = (
            EXPECTED_DATASETS[
                dataset_key
            ]
        )

        cfg[
            "experiment"
        ]["seed"] = seed


        if (
            "epochs"
            in cfg["experiment"]
        ):

            cfg[
                "experiment"
            ]["epochs"] = 400


        # ====================================================
        # Training
        # ====================================================

        cfg[
            "training"
        ]["epochs"] = 400


        if (
            "seed"
            in cfg["training"]
        ):

            cfg[
                "training"
            ]["seed"] = seed


        # Fresh formal runs:
        # never inherit sensitivity resume paths.
        cfg[
            "training"
        ].pop(
            "resume_from",
            None,
        )


        # ====================================================
        # Kaggle data path
        # ====================================================

        cfg[
            "data"
        ]["root"] = str(
            DATA_ROOT
        )


        # ====================================================
        # Independent final output root
        # ====================================================

        cfg[
            "output"
        ]["root"] = (
            FINAL_RESULT_ROOT_NAME
        )


        # ====================================================
        # Freeze official clustering protocol
        # ====================================================

        cfg[
            "clustering"
        ]["method"] = (
            "kmeans"
        )

        cfg[
            "clustering"
        ]["embedding"] = (
            "concat_z"
        )

        cfg[
            "clustering"
        ]["n_clusters"] = (
            EXPECTED_CLUSTERS[
                dataset_key
            ]
        )

        cfg[
            "clustering"
        ]["n_init"] = 20

        cfg[
            "clustering"
        ]["random_state"] = 0


        # ====================================================
        # Freeze BSRR
        # ====================================================

        cfg[
            "refinement"
        ]["enabled"] = True

        cfg[
            "refinement"
        ]["method"] = (
            "bsrr"
        )

        cfg[
            "refinement"
        ]["spatial_k"] = 3


        # ====================================================
        # NMI
        # ====================================================

        cfg[
            "evaluation"
        ][
            "nmi_average_method"
        ] = "max"


        # ====================================================
        # Save YAML
        # ====================================================

        config_path = (
            FINAL_CONFIG_DIR
            / f"{experiment_name}.yaml"
        )


        with config_path.open(
            "w",
            encoding="utf-8",
        ) as f:

            yaml.safe_dump(
                cfg,
                f,
                sort_keys=False,
                allow_unicode=True,
            )


        FINAL_CONFIGS.append(
            config_path
        )


# ============================================================
# Final check
# ============================================================

assert len(
    FINAL_CONFIGS
) == 50


print("=" * 90)
print("FORMAL 400-EPOCH CONFIG GENERATION")
print("=" * 90)


for dataset_key in (
    BASE_CONFIGS
):

    paths = [
        p
        for p in FINAL_CONFIGS
        if p.name.startswith(
            dataset_key
            + "_"
        )
    ]

    print(
        f"{dataset_key:7s}: "
        f"{len(paths)} configs"
    )

    assert len(paths) == 10


print(
    "\nTotal configs:",
    len(FINAL_CONFIGS),
)


print(
    "\nConfig directory:"
)

print(
    FINAL_CONFIG_DIR
)


print(
    "\nPASS: 50 formal 400-epoch "
    "configs generated."
)

FORMAL 400-EPOCH CONFIG GENERATION
hlna1  : 10 configs
d1     : 10 configs
e185   : 10 configs
s2e15  : 10 configs
s2e18  : 10 configs

Total configs: 50

Config directory:
/kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep

PASS: 50 formal 400-epoch configs generated.


Cell 7：完整审计 50 个正式配置

In [7]:
# ============================================================
# Cell 7
# Audit all 50 formal configs
# ============================================================

import pandas as pd
import yaml


audit_rows = []


for config_path in sorted(
    FINAL_CONFIGS
):

    with config_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        cfg = yaml.safe_load(f)


    experiment = (
        cfg["experiment"]
    )

    training = (
        cfg["training"]
    )

    clustering = (
        cfg["clustering"]
    )

    refinement = (
        cfg["refinement"]
    )

    evaluation = (
        cfg["evaluation"]
    )


    dataset = (
        experiment[
            "dataset"
        ]
    )

    seed = int(
        experiment[
            "seed"
        ]
    )

    epochs = int(
        training[
            "epochs"
        ]
    )

    warm_up = int(
        training[
            "warm_up_epochs"
        ]
    )

    n_clusters = int(
        clustering[
            "n_clusters"
        ]
    )


    audit_rows.append(
        {
            "dataset":
                dataset,

            "seed":
                seed,

            "epochs":
                epochs,

            "warm_up":
                warm_up,

            "K":
                n_clusters,

            "embedding":
                clustering[
                    "embedding"
                ],

            "method":
                clustering[
                    "method"
                ],

            "n_init":
                int(
                    clustering[
                        "n_init"
                    ]
                ),

            "kmeans_rs":
                int(
                    clustering[
                        "random_state"
                    ]
                ),

            "refinement":
                refinement[
                    "method"
                ],

            "spatial_k":
                int(
                    refinement[
                        "spatial_k"
                    ]
                ),

            "nmi_method":
                evaluation[
                    "nmi_average_method"
                ],

            "data_root":
                cfg["data"][
                    "root"
                ],

            "output_root":
                cfg["output"][
                    "root"
                ],

            "resume_from":
                training.get(
                    "resume_from",
                    None,
                ),
        }
    )


audit_df = pd.DataFrame(
    audit_rows
)


print(
    audit_df.to_string(
        index=False
    )
)


# ============================================================
# Global assertions
# ============================================================

assert len(
    audit_df
) == 50


assert (
    audit_df[
        "epochs"
    ] == 400
).all()


assert (
    audit_df[
        "warm_up"
    ] == 10
).all()


assert (
    audit_df[
        "embedding"
    ].str.lower()
    == "concat_z"
).all()


assert (
    audit_df[
        "method"
    ].str.lower()
    == "kmeans"
).all()


assert (
    audit_df[
        "n_init"
    ] == 20
).all()


assert (
    audit_df[
        "kmeans_rs"
    ] == 0
).all()


assert (
    audit_df[
        "refinement"
    ].str.lower()
    == "bsrr"
).all()


assert (
    audit_df[
        "spatial_k"
    ] == 3
).all()


assert (
    audit_df[
        "nmi_method"
    ] == "max"
).all()


assert (
    audit_df[
        "data_root"
    ]
    == str(DATA_ROOT)
).all()


assert (
    audit_df[
        "output_root"
    ]
    == "results_final_400"
).all()


# Formal runs MUST start fresh.
assert (
    audit_df[
        "resume_from"
    ].isna()
).all()


# ============================================================
# Dataset × seed completeness
# ============================================================

expected_datasets = {

    "HLN-A1": 10,
    "HLN-D1": 11,
    "E18.5": 14,
    "S2-E15": 15,
    "S2-E18": 16,
}


for dataset, expected_k in (
    expected_datasets.items()
):

    d = audit_df[
        audit_df[
            "dataset"
        ] == dataset
    ]


    assert len(d) == 10, (
        f"{dataset}: expected "
        f"10 configs, got {len(d)}"
    )


    assert (
        sorted(
            d["seed"].tolist()
        )
        == list(
            range(10)
        )
    ), (
        f"{dataset}: seed set "
        f"is not 0..9"
    )


    assert (
        d["K"]
        == expected_k
    ).all(), (
        f"{dataset}: wrong K"
    )


print(
    "\n" + "=" * 90
)

print(
    "PASS: FORMAL 400-EPOCH "
    "PROTOCOL FROZEN"
)

print(
    "=" * 90
)

print(
    "\nDatasets : 5"
)

print(
    "Seeds    : 0-9"
)

print(
    "Runs     : 50"
)

print(
    "Epochs   : 400"
)

print(
    "Warm-up  : 10"
)

print(
    "Readout  : concat-Z -> BSRR -> KMeans"
)

print(
    "BSRR k   : 3"
)

print(
    "KMeans   : n_init=20, random_state=0"
)

print(
    "NMI      : average_method=max"
)

print(
    "Output   : results_final_400"
)

print(
    "\nDO NOT START TRAINING YET."
)

dataset  seed  epochs  warm_up  K embedding method  n_init  kmeans_rs refinement  spatial_k nmi_method                               data_root       output_root resume_from
 HLN-D1     0     400       10 11  concat_z kmeans      20          0       bsrr          3        max /kaggle/input/datasets/wuvdji/smgc-data results_final_400        None
 HLN-D1     1     400       10 11  concat_z kmeans      20          0       bsrr          3        max /kaggle/input/datasets/wuvdji/smgc-data results_final_400        None
 HLN-D1     2     400       10 11  concat_z kmeans      20          0       bsrr          3        max /kaggle/input/datasets/wuvdji/smgc-data results_final_400        None
 HLN-D1     3     400       10 11  concat_z kmeans      20          0       bsrr          3        max /kaggle/input/datasets/wuvdji/smgc-data results_final_400        None
 HLN-D1     4     400       10 11  concat_z kmeans      20          0       bsrr          3        max /kaggle/input/datasets/wuvdji/sm

## Cell 8：创建正式 400-epoch checkpoint/resume runner

This runner preserves the original SpaMGCL training and evaluation protocol,
while adding checkpoint-based recovery for long 400-epoch formal runs.

Recovery checkpoints are saved at epochs 100, 200, 300, and 400.
These checkpoints are used only for interruption recovery and are not used
for model selection.

In [8]:
# ============================================================
# Cell 8
# Create formal 400-epoch checkpoint/resume runner
# ============================================================

from pathlib import Path
import re
import subprocess
import sys


PROJECT_ROOT = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL"
)

RUNNER_ORIGINAL = (
    PROJECT_ROOT
    / "experiments"
    / "run_exp.py"
)

RUNNER_FORMAL = (
    PROJECT_ROOT
    / "experiments"
    / "run_exp_formal_400.py"
)


assert RUNNER_ORIGINAL.exists(), (
    RUNNER_ORIGINAL
)


# ============================================================
# Read original runner
# ============================================================

source = RUNNER_ORIGINAL.read_text(
    encoding="utf-8"
)

patched = source


# ============================================================
# Safety check:
# original runner must already support checkpoint RNG state
# ============================================================

required_checkpoint_terms = [
    "python_rng_state",
    "numpy_rng_state",
    "torch_rng_state",
    "checkpoint_epoch",
]

for term in required_checkpoint_terms:

    assert term in source, (
        f"Original runner does not contain "
        f"checkpoint field: {term}"
    )


# ============================================================
# Patch 1
# Formal recovery milestones:
# 100 / 200 / 300 / 400
# ============================================================

milestone_pattern = re.compile(
    r"milestone_epochs\s*=\s*"
    r"\{\s*50\s*,\s*100\s*,\s*200\s*\}"
)

matches = list(
    milestone_pattern.finditer(
        patched
    )
)

print(
    "Original milestone definitions found:",
    len(matches),
)


assert len(matches) == 1, (
    "STOP: expected exactly one "
    "milestone_epochs = {50, 100, 200} "
    "definition in run_exp.py."
)


patched = milestone_pattern.sub(
    (
        "milestone_epochs = "
        "{100, 200, 300, 400}"
    ),
    patched,
    count=1,
)


# ============================================================
# Patch 2
# Insert exact resume logic
# ============================================================

marker = (
    '    print(f"Dataset: {dataset} | '
    'spots={sample.n_spots} | device={device}")'
)


assert marker in patched, (
    "STOP: cannot locate model/data "
    "initialization marker."
)


resume_block = r'''
    # ------------------------------------------------------------
    # Optional exact resume from formal checkpoint.
    #
    # The checkpoint restores:
    # - model state
    # - optimizer state
    # - Python RNG
    # - NumPy RNG
    # - Torch CPU RNG
    # - Torch CUDA RNG
    # - previous loss history
    #
    # resume_from is used ONLY for interruption recovery.
    # ------------------------------------------------------------

    resume_from_value = training_config.get(
        "resume_from"
    )

    resume_checkpoint_path = None
    start_epoch = 0
    resume_history = []

    if resume_from_value:

        resume_checkpoint_path = _resolve_path(
            resume_from_value,
            base=PROJECT_ROOT,
            field_name="training.resume_from",
        )

        if not resume_checkpoint_path.is_file():

            raise FileNotFoundError(
                "Resume checkpoint not found: "
                f"{resume_checkpoint_path}"
            )


        checkpoint = torch.load(
            resume_checkpoint_path,
            map_location=device,
            weights_only=False,
        )


        required_checkpoint_keys = {
            "model",
            "optimizer",
            "epoch",
            "python_rng_state",
            "numpy_rng_state",
            "torch_rng_state",
        }


        missing_checkpoint_keys = (
            required_checkpoint_keys
            - set(checkpoint)
        )


        if missing_checkpoint_keys:

            raise KeyError(
                "Resume checkpoint is missing keys: "
                f"{sorted(missing_checkpoint_keys)}"
            )


        # --------------------------------------------------------
        # Restore model + optimizer
        # --------------------------------------------------------

        model.load_state_dict(
            checkpoint["model"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer"]
        )


        start_epoch = int(
            checkpoint["epoch"]
        )


        # start_epoch == epochs is allowed:
        # this supports the rare case where epoch-400 checkpoint
        # exists but final metrics.json was not written.
        if (
            start_epoch < 0
            or start_epoch > epochs
        ):

            raise ValueError(
                "Resume epoch must satisfy "
                "0 <= resume_epoch <= epochs; "
                f"resume_epoch={start_epoch}, "
                f"epochs={epochs}"
            )


        # --------------------------------------------------------
        # Restore Python / NumPy RNG
        # --------------------------------------------------------

        random.setstate(
            checkpoint[
                "python_rng_state"
            ]
        )


        np.random.set_state(
            checkpoint[
                "numpy_rng_state"
            ]
        )


        # --------------------------------------------------------
        # Normalize PyTorch RNG representation
        # --------------------------------------------------------

        def _normalize_torch_rng_state(
            state,
            state_name,
        ):

            if torch.is_tensor(state):

                normalized = (
                    state
                    .detach()
                    .cpu()
                    .to(
                        dtype=torch.uint8
                    )
                    .contiguous()
                )

            elif isinstance(
                state,
                np.ndarray,
            ):

                normalized = (
                    torch.from_numpy(
                        np.asarray(
                            state,
                            dtype=np.uint8,
                        )
                    )
                    .cpu()
                    .contiguous()
                )

            elif isinstance(
                state,
                (list, tuple),
            ):

                normalized = torch.tensor(
                    state,
                    dtype=torch.uint8,
                    device="cpu",
                ).contiguous()

            else:

                raise TypeError(
                    f"Unsupported "
                    f"{state_name} type: "
                    f"{type(state)}"
                )

            return normalized


        # --------------------------------------------------------
        # Restore Torch CPU RNG
        # --------------------------------------------------------

        torch_rng_state = (
            _normalize_torch_rng_state(
                checkpoint[
                    "torch_rng_state"
                ],
                "torch_rng_state",
            )
        )


        torch.set_rng_state(
            torch_rng_state
        )


        # --------------------------------------------------------
        # Restore CUDA RNG
        # --------------------------------------------------------

        if (
            torch.cuda.is_available()
            and
            "cuda_rng_state"
            in checkpoint
        ):

            cuda_states = checkpoint[
                "cuda_rng_state"
            ]


            # A single tensor/array
            if (
                torch.is_tensor(
                    cuda_states
                )
                or isinstance(
                    cuda_states,
                    np.ndarray,
                )
            ):

                cuda_states = [
                    cuda_states
                ]


            elif isinstance(
                cuda_states,
                (list, tuple),
            ):

                if len(
                    cuda_states
                ) == 0:

                    raise RuntimeError(
                        "cuda_rng_state is empty"
                    )


                # Flat list of integers:
                # one single CUDA RNG state
                if isinstance(
                    cuda_states[0],
                    (int, np.integer),
                ):

                    cuda_states = [
                        cuda_states
                    ]

            else:

                raise TypeError(
                    "Unsupported "
                    "cuda_rng_state type: "
                    f"{type(cuda_states)}"
                )


            normalized_cuda_states = [

                _normalize_torch_rng_state(
                    state,
                    f"cuda_rng_state[{i}]",
                )

                for i, state
                in enumerate(
                    cuda_states
                )
            ]


            torch.cuda.set_rng_state_all(
                normalized_cuda_states
            )


        # --------------------------------------------------------
        # Restore loss history
        # --------------------------------------------------------

        history_path = (
            resume_checkpoint_path.parent
            / (
                f"metrics_epoch"
                f"{start_epoch}.json"
            )
        )


        if not history_path.is_file():

            raise FileNotFoundError(
                "Cannot reconstruct "
                "pre-resume loss history: "
                f"{history_path}"
            )


        history_snapshot = json.loads(
            history_path.read_text(
                encoding="utf-8"
            )
        )


        resume_history = list(
            history_snapshot.get(
                "loss_history",
                [],
            )
        )


        if (
            len(resume_history)
            != start_epoch
        ):

            raise RuntimeError(
                "Resume history length "
                "does not match checkpoint epoch: "
                f"len(history)="
                f"{len(resume_history)}, "
                f"checkpoint_epoch="
                f"{start_epoch}"
            )


        if resume_history:

            last_history_epoch = int(
                resume_history[-1][
                    "epoch"
                ]
            )


            if (
                last_history_epoch
                != start_epoch
            ):

                raise RuntimeError(
                    "Resume history ends "
                    "at wrong epoch: "
                    f"{last_history_epoch} "
                    f"!= {start_epoch}"
                )


        print(
            "\nRESUME MODE:"
        )

        print(
            "Checkpoint:",
            resume_checkpoint_path,
        )

        print(
            "Resume epoch:",
            start_epoch,
        )

'''


patched = patched.replace(
    marker,
    resume_block + marker,
    1,
)


# ============================================================
# Patch 3
# Start training loop from start_epoch
# ============================================================

old_loop = '''    history = []
    cluster_head_init = str(training_config.get("cluster_head_init", "none")).lower()
    cluster_head_initialized = False
    for epoch_index in range(epochs):
'''


new_loop = '''    history = list(resume_history)
    cluster_head_init = str(training_config.get("cluster_head_init", "none")).lower()
    cluster_head_initialized = (
        start_epoch >= warm_up_epochs
        and cluster_head_init == "kmeans"
    )
    for epoch_index in range(start_epoch, epochs):
'''


assert old_loop in patched, (
    "STOP: training loop pattern "
    "not found."
)


patched = patched.replace(
    old_loop,
    new_loop,
    1,
)


# ============================================================
# Save separate formal runner
# ============================================================

RUNNER_FORMAL.write_text(
    patched,
    encoding="utf-8",
)


# ============================================================
# Syntax check
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(RUNNER_FORMAL),
    ],
    check=True,
)


# ============================================================
# Final source checks
# ============================================================

formal_text = (
    RUNNER_FORMAL.read_text(
        encoding="utf-8"
    )
)


assert (
    "milestone_epochs = "
    "{100, 200, 300, 400}"
    in formal_text
)

assert (
    "range(start_epoch, epochs)"
    in formal_text
)

assert (
    "_normalize_torch_rng_state"
    in formal_text
)

assert (
    "resume_from_value"
    in formal_text
)


print("=" * 90)
print("FORMAL RUNNER CREATED")
print("=" * 90)

print(
    "Original:",
    RUNNER_ORIGINAL,
)

print(
    "Formal  :",
    RUNNER_FORMAL,
)

print(
    "\nRecovery checkpoints:"
)

print(
    "100 / 200 / 300 / 400"
)

print(
    "\nPASS: formal 400-epoch "
    "resume runner created."
)

Original milestone definitions found: 1
FORMAL RUNNER CREATED
Original: /kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp.py
Formal  : /kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp_formal_400.py

Recovery checkpoints:
100 / 200 / 300 / 400

PASS: formal 400-epoch resume runner created.


## Cell 9：审计正式 runner，不启动训练

In [9]:
# ============================================================
# Cell 9
# Audit formal runner
# ============================================================

import re
import subprocess
import sys


text = RUNNER_FORMAL.read_text(
    encoding="utf-8"
)


checks = {

    "400 milestones":
        (
            "milestone_epochs = "
            "{100, 200, 300, 400}"
            in text
        ),

    "resume config":
        (
            "resume_from_value"
            in text
        ),

    "resume loop":
        (
            "range(start_epoch, epochs)"
            in text
        ),

    "model restore":
        (
            'model.load_state_dict('
            in text
        ),

    "optimizer restore":
        (
            'optimizer.load_state_dict('
            in text
        ),

    "Python RNG":
        (
            "random.setstate("
            in text
        ),

    "NumPy RNG":
        (
            "np.random.set_state("
            in text
        ),

    "Torch RNG":
        (
            "torch.set_rng_state("
            in text
        ),

    "CUDA RNG":
        (
            "torch.cuda.set_rng_state_all("
            in text
        ),

    "RNG normalization":
        (
            "_normalize_torch_rng_state"
            in text
        ),
}


print("=" * 80)
print("FORMAL RUNNER AUDIT")
print("=" * 80)


for name, passed in (
    checks.items()
):

    print(
        f"{'PASS' if passed else 'FAIL':4s}"
        f" | {name}"
    )


assert all(
    checks.values()
)


subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(RUNNER_FORMAL),
    ],
    check=True,
)


print(
    "\nPASS: formal runner audit complete."
)

print(
    "NO TRAINING HAS BEEN STARTED."
)

FORMAL RUNNER AUDIT
PASS | 400 milestones
PASS | resume config
PASS | resume loop
PASS | model restore
PASS | optimizer restore
PASS | Python RNG
PASS | NumPy RNG
PASS | Torch RNG
PASS | CUDA RNG
PASS | RNG normalization

PASS: formal runner audit complete.
NO TRAINING HAS BEEN STARTED.


## Cell 10：定义正式实验自动恢复 + 审计函数

In [10]:
# ============================================================
# Cell 10
# Formal run / resume / audit controller
# ============================================================

from pathlib import Path
import copy
import json
import shutil
import subprocess
import sys
import yaml


PROJECT_ROOT = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL"
)

FINAL_RESULT_ROOT = (
    PROJECT_ROOT
    / "results_final_400"
)

RESUME_CONFIG_DIR = (
    PROJECT_ROOT
    / "configs"
    / "formal_400ep_resume"
)

RESUME_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


RECOVERY_EPOCHS = [
    400,
    300,
    200,
    100,
]


FINAL_REQUIRED_FILES = [
    "metrics.json",
    "config.yaml",
    "manifest.json",
    "pred_labels.npy",
    "pred_concat_z_kmeans.npy",
    "pred_q_argmax.npy",
    "gt_labels.npy",
    "coords.npy",
    "z_concat.npy",
    "z_bsrr.npy",
    "checkpoint_last.pt",
    "metrics_epoch400.json",
    "checkpoint_epoch400.pt",
]


def run_formal_config(
    config_path,
):

    config_path = Path(
        config_path
    )


    assert config_path.exists(), (
        config_path
    )


    # ========================================================
    # Read formal config
    # ========================================================

    with config_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        cfg = yaml.safe_load(f)


    exp_name = (
        cfg["experiment"][
            "name"
        ]
    )

    dataset = (
        cfg["experiment"][
            "dataset"
        ]
    )

    seed = int(
        cfg["experiment"][
            "seed"
        ]
    )


    assert int(
        cfg["training"][
            "epochs"
        ]
    ) == 400


    assert (
        cfg["training"].get(
            "resume_from"
        )
        is None
    )


    output_root = Path(
        cfg["output"][
            "root"
        ]
    )


    if not output_root.is_absolute():

        output_root = (
            PROJECT_ROOT
            / output_root
        )


    output_dir = (
        output_root
        / exp_name
    )


    final_metrics = (
        output_dir
        / "metrics.json"
    )


    print(
        "\n" + "=" * 90
    )

    print(
        "FORMAL EXPERIMENT"
    )

    print(
        "=" * 90
    )

    print(
        "Dataset :",
        dataset,
    )

    print(
        "Seed    :",
        seed,
    )

    print(
        "Name    :",
        exp_name,
    )

    print(
        "Epochs  : 400"
    )

    print(
        "Config  :",
        config_path,
    )

    print(
        "Output  :",
        output_dir,
    )

    print(
        "=" * 90
    )


    # ========================================================
    # Case 1:
    # Complete formal result
    # ========================================================

    if final_metrics.exists():

        print(
            "\nFinal metrics already exist."
        )

        print(
            "Skipping training and "
            "running audit only."
        )


        run_config_path = (
            config_path
        )


    else:

        # ====================================================
        # Search for latest valid recovery checkpoint
        # ====================================================

        resume_epoch = None
        resume_checkpoint = None


        for epoch in (
            RECOVERY_EPOCHS
        ):

            checkpoint_path = (
                output_dir
                / (
                    f"checkpoint_epoch"
                    f"{epoch}.pt"
                )
            )

            metrics_epoch_path = (
                output_dir
                / (
                    f"metrics_epoch"
                    f"{epoch}.json"
                )
            )


            # One exists without the other:
            # do not guess.
            if (
                checkpoint_path.exists()
                !=
                metrics_epoch_path.exists()
            ):

                raise RuntimeError(
                    "\nIncomplete recovery pair "
                    f"for epoch {epoch}:\n"
                    f"{checkpoint_path}\n"
                    f"{metrics_epoch_path}"
                )


            if (
                checkpoint_path.exists()
                and
                metrics_epoch_path.exists()
            ):

                resume_epoch = epoch

                resume_checkpoint = (
                    checkpoint_path
                )

                break


        # ====================================================
        # Case 2:
        # Resume
        # ====================================================

        if (
            resume_checkpoint
            is not None
        ):

            print(
                "\nRecovery checkpoint detected:"
            )

            print(
                resume_checkpoint
            )

            print(
                "Resume from epoch:",
                resume_epoch,
            )


            resume_cfg = copy.deepcopy(
                cfg
            )


            resume_cfg[
                "training"
            ][
                "resume_from"
            ] = str(
                resume_checkpoint
            )


            resume_cfg[
                "training"
            ][
                "epochs"
            ] = 400


            if (
                "epochs"
                in resume_cfg[
                    "experiment"
                ]
            ):

                resume_cfg[
                    "experiment"
                ][
                    "epochs"
                ] = 400


            run_config_path = (
                RESUME_CONFIG_DIR
                / (
                    f"{exp_name}"
                    f"_resume"
                    f"{resume_epoch}.yaml"
                )
            )


            with run_config_path.open(
                "w",
                encoding="utf-8",
            ) as f:

                yaml.safe_dump(
                    resume_cfg,
                    f,
                    sort_keys=False,
                    allow_unicode=True,
                )


            print(
                "Temporary resume config:"
            )

            print(
                run_config_path
            )


        # ====================================================
        # Case 3:
        # Fresh run
        # ====================================================

        else:

            # ----------------------------------------------
            # Check for unsafe partial output
            # ----------------------------------------------

            if output_dir.exists():

                existing_files = list(
                    output_dir.iterdir()
                )


                trained_files = [

                    p
                    for p in existing_files

                    if (
                        p.name.startswith(
                            "metrics_epoch"
                        )
                        or
                        p.name.startswith(
                            "checkpoint_epoch"
                        )
                        or
                        p.name
                        == "checkpoint_last.pt"
                    )
                ]


                if trained_files:

                    print(
                        "\nPartial trained "
                        "directory detected:"
                    )

                    for p in sorted(
                        existing_files
                    ):

                        print(
                            " -",
                            p.name,
                        )


                    raise RuntimeError(
                        "\nSTOP: partial training "
                        "exists but no valid "
                        "recovery checkpoint pair "
                        "was found."
                    )


                # Only harmless pre-training residue
                if existing_files:

                    print(
                        "\nRemoving pre-training "
                        "directory residue:"
                    )

                    for p in sorted(
                        existing_files
                    ):

                        print(
                            " -",
                            p.name,
                        )


                    shutil.rmtree(
                        output_dir
                    )


            print(
                "\nStarting fresh formal run."
            )


            run_config_path = (
                config_path
            )


        # ====================================================
        # Execute training / resume
        # ====================================================

        cmd = [
            sys.executable,
            str(
                RUNNER_FORMAL
            ),
            "--config",
            str(
                run_config_path
            ),
        ]


        print(
            "\nRunning:"
        )

        print(
            " ".join(cmd)
        )

        print()


        subprocess.run(
            cmd,
            cwd=PROJECT_ROOT,
            check=True,
        )


    # ========================================================
    # Verify final output
    # ========================================================

    print(
        "\nChecking final artifacts..."
    )


    missing = []


    for filename in (
        FINAL_REQUIRED_FILES
    ):

        p = (
            output_dir
            / filename
        )

        status = (
            "OK"
            if p.exists()
            else "MISSING"
        )

        print(
            f"{status:7s} "
            f"{filename}"
        )


        if not p.exists():

            missing.append(
                filename
            )


    assert not missing, (
        f"Missing final files: "
        f"{missing}"
    )


    # ========================================================
    # Official audit
    # ========================================================

    print(
        "\nRunning official audit..."
    )


    subprocess.run(
        [
            sys.executable,
            str(
                PROJECT_ROOT
                / "scripts"
                / "audit_run.py"
            ),
            str(
                output_dir
            ),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )


    # ========================================================
    # Read final metrics
    # ========================================================

    with (
        output_dir
        / "metrics.json"
    ).open(
        "r",
        encoding="utf-8",
    ) as f:

        metrics = json.load(f)


    print(
        "\n" + "=" * 90
    )

    print(
        "FORMAL RUN COMPLETE"
    )

    print(
        "=" * 90
    )

    print(
        "Dataset:",
        dataset,
    )

    print(
        "Seed:",
        seed,
    )

    print(
        f"ARI = "
        f"{float(metrics['ARI']):.6f}"
    )

    print(
        f"NMI = "
        f"{float(metrics['NMI']):.6f}"
    )

    print(
        "=" * 90
    )


    return output_dir

## Cell 11：正式运行 HLN-A1 seed 0

In [11]:
# ============================================================
# Cell 11
# First formal run:
# HLN-A1 | seed 0 | 400 epochs
# ============================================================

HLNA1_SEED0_CONFIG = (
    FINAL_CONFIG_DIR
    / "hlna1_formal_400ep_seed0.yaml"
)


assert HLNA1_SEED0_CONFIG.exists(), (
    HLNA1_SEED0_CONFIG
)


hlna1_seed0_dir = (
    run_formal_config(
        HLNA1_SEED0_CONFIG
    )
)


print(
    "\nPASS: HLN-A1 seed0 "
    "formal 400-epoch run completed."
)


FORMAL EXPERIMENT
Dataset : HLN-A1
Seed    : 0
Name    : hlna1_formal_400ep_seed0
Epochs  : 400
Config  : /kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep/hlna1_formal_400ep_seed0.yaml
Output  : /kaggle/working/SpaMGCL/SpaMGCL/results_final_400/hlna1_formal_400ep_seed0

Starting fresh formal run.

Running:
/usr/bin/python3 /kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp_formal_400.py --config /kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep/hlna1_formal_400ep_seed0.yaml



/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.411675 | rec=0.252383 | mgcl=8.159292 | cluster=2.951158 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.887e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/400 | total=8.400130 | rec=0.242314 | mgcl=8.157817 | cluster=2.950293 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.620e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.956 | gradC=0.000e+00
epoch 003/400 | total=8.389496 | rec=0.232629 | mgcl=8.156867 | cluster=2.949508 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.570e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.962 | gradC=0.000e+00
epoch 00

Cell 12：独立核验 seed0 正式结果

In [12]:
# ============================================================
# Cell 12
# Independent verification of HLN-A1 seed0
# ============================================================

import json
import numpy as np

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


RUN_DIR = (
    FINAL_RESULT_ROOT
    / "hlna1_formal_400ep_seed0"
)


with (
    RUN_DIR
    / "metrics.json"
).open(
    "r",
    encoding="utf-8",
) as f:

    metrics = json.load(f)


gt = np.load(
    RUN_DIR
    / "gt_labels.npy"
)

pred = np.load(
    RUN_DIR
    / "pred_labels.npy"
)

pred_raw = np.load(
    RUN_DIR
    / "pred_concat_z_kmeans.npy"
)

z_concat = np.load(
    RUN_DIR
    / "z_concat.npy"
)

z_bsrr = np.load(
    RUN_DIR
    / "z_bsrr.npy"
)


ari_check = (
    adjusted_rand_score(
        gt,
        pred,
    )
)

nmi_check = (
    normalized_mutual_info_score(
        gt,
        pred,
        average_method="max",
    )
)


raw_ari = (
    adjusted_rand_score(
        gt,
        pred_raw,
    )
)

raw_nmi = (
    normalized_mutual_info_score(
        gt,
        pred_raw,
        average_method="max",
    )
)


print("=" * 80)
print("HLN-A1 SEED0 FINAL VERIFICATION")
print("=" * 80)

print(
    "GT spots:",
    len(gt),
)

print(
    "z_concat:",
    z_concat.shape,
)

print(
    "z_bsrr  :",
    z_bsrr.shape,
)


print(
    "\nOfficial metrics.json"
)

print(
    f"ARI = "
    f"{float(metrics['ARI']):.12f}"
)

print(
    f"NMI = "
    f"{float(metrics['NMI']):.12f}"
)


print(
    "\nIndependent recomputation"
)

print(
    f"ARI = "
    f"{ari_check:.12f}"
)

print(
    f"NMI = "
    f"{nmi_check:.12f}"
)


print(
    "\nRaw concat-Z diagnostic"
)

print(
    f"ARI = "
    f"{raw_ari:.12f}"
)

print(
    f"NMI = "
    f"{raw_nmi:.12f}"
)


print(
    "\nBSRR change"
)

print(
    f"Delta ARI = "
    f"{ari_check - raw_ari:+.12f}"
)

print(
    f"Delta NMI = "
    f"{nmi_check - raw_nmi:+.12f}"
)


assert np.isclose(
    ari_check,
    float(
        metrics["ARI"]
    ),
)

assert np.isclose(
    nmi_check,
    float(
        metrics["NMI"]
    ),
)


assert (
    len(gt)
    == z_concat.shape[0]
    == z_bsrr.shape[0]
)


print(
    "\nPASS: HLN-A1 seed0 "
    "formal result independently verified."
)

HLN-A1 SEED0 FINAL VERIFICATION
GT spots: 3484
z_concat: (3484, 128)
z_bsrr  : (3484, 128)

Official metrics.json
ARI = 0.220614168388
NMI = 0.358325106734

Independent recomputation
ARI = 0.220614168388
NMI = 0.358325106734

Raw concat-Z diagnostic
ARI = 0.218924449036
NMI = 0.353420810942

BSRR change
Delta ARI = +0.001689719353
Delta NMI = +0.004904295792

PASS: HLN-A1 seed0 formal result independently verified.


Cell 13：运行 HLN-A1 剩余 seeds 1–9

In [13]:
# ============================================================
# Cell 13
# Run remaining HLN-A1 formal seeds
# ============================================================

from pathlib import Path
import time


# 如果担心 Kaggle 单次 session 时间，
# 可以先改成 [1, 2, 3, 4]，
# 下一次再改成 [5, 6, 7, 8, 9]
SEEDS_TO_RUN = list(range(1, 10))


completed_dirs = []


print("=" * 90)
print("HLN-A1 FORMAL 400-EPOCH RUNS")
print("=" * 90)

print("Seeds to run:", SEEDS_TO_RUN)
print("Total:", len(SEEDS_TO_RUN))


for i, seed in enumerate(
    SEEDS_TO_RUN,
    start=1,
):

    config_path = (
        FINAL_CONFIG_DIR
        / f"hlna1_formal_400ep_seed{seed}.yaml"
    )

    assert config_path.exists(), (
        f"Missing config: {config_path}"
    )


    print("\n")
    print("#" * 90)

    print(
        f"HLN-A1 seed {seed}"
        f" | {i}/{len(SEEDS_TO_RUN)}"
    )

    print("#" * 90)


    start_time = time.time()


    output_dir = run_formal_config(
        config_path
    )


    elapsed_min = (
        time.time() - start_time
    ) / 60.0


    completed_dirs.append(
        output_dir
    )


    print(
        f"\nSeed {seed} finished "
        f"in {elapsed_min:.2f} min"
    )


print("\n" + "=" * 90)
print("HLN-A1 REQUESTED SEEDS COMPLETED")
print("=" * 90)

print(
    "Completed in this call:",
    len(completed_dirs),
)

for p in completed_dirs:
    print(" -", p.name)

HLN-A1 FORMAL 400-EPOCH RUNS
Seeds to run: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Total: 9


##########################################################################################
HLN-A1 seed 1 | 1/9
##########################################################################################

FORMAL EXPERIMENT
Dataset : HLN-A1
Seed    : 1
Name    : hlna1_formal_400ep_seed1
Epochs  : 400
Config  : /kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep/hlna1_formal_400ep_seed1.yaml
Output  : /kaggle/working/SpaMGCL/SpaMGCL/results_final_400/hlna1_formal_400ep_seed1

Starting fresh formal run.

Running:
/usr/bin/python3 /kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp_formal_400.py --config /kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep/hlna1_formal_400ep_seed1.yaml



/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.396720 | rec=0.237463 | mgcl=8.159257 | cluster=2.950006 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.644e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.959 | gradC=0.000e+00
epoch 002/400 | total=8.386283 | rec=0.228770 | mgcl=8.157513 | cluster=2.949127 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.560e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.966 | gradC=0.000e+00
epoch 003/400 | total=8.376886 | rec=0.220448 | mgcl=8.156439 | cluster=2.948355 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.735e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.972 | gradC=0.000e+00
epoch 00

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.412103 | rec=0.254117 | mgcl=8.157986 | cluster=2.951155 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=9.127e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/400 | total=8.401219 | rec=0.244371 | mgcl=8.156848 | cluster=2.950457 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.164e-02 | neg_count=12134772 | snf_masked_positions=0 | effC=9.955 | gradC=0.000e+00
epoch 003/400 | total=8.391184 | rec=0.235091 | mgcl=8.156094 | cluster=2.949790 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.281e-02 | neg_count=12134772 | snf_masked_positions=0 | effC=9.959 | gradC=0.000e+00
epoch 00

...

Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.447705 | rec=0.289945 | mgcl=8.157760 | cluster=2.951674 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.613e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/400 | total=8.435236 | rec=0.278373 | mgcl=8.156863 | cluster=2.950777 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.174e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.957 | gradC=0.000e+00
epoch 003/400 | total=8.423660 | rec=0.267463 | mgcl=8.156198 | cluster=2.949970 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.050e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.962 | gradC=0.000e+00
epoch 00

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.417413 | rec=0.259766 | mgcl=8.157647 | cluster=2.950842 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.794e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.953 | gradC=0.000e+00
epoch 002/400 | total=8.405575 | rec=0.249104 | mgcl=8.156470 | cluster=2.949901 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=9.827e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.961 | gradC=0.000e+00
epoch 003/400 | total=8.394567 | rec=0.239003 | mgcl=8.155563 | cluster=2.949053 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.031e-02 | neg_count=12134772 | snf_masked_positions=0 | effC=9.967 | gradC=0.000e+00
epoch 00

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.392878 | rec=0.234356 | mgcl=8.158521 | cluster=2.951650 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.590e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.936 | gradC=0.000e+00
epoch 002/400 | total=8.382640 | rec=0.225641 | mgcl=8.157000 | cluster=2.950634 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.190e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.945 | gradC=0.000e+00
epoch 003/400 | total=8.373121 | rec=0.217257 | mgcl=8.155864 | cluster=2.949736 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.015e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.953 | gradC=0.000e+00
epoch 00

Cell 14：汇总 HLN-A1 的 10 seeds

In [14]:
# ============================================================
# Cell 14
# HLN-A1 10-seed final summary
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


SUMMARY_DIR = (
    PROJECT_ROOT
    / "formal_summary_400"
)

SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


rows = []


for seed in range(10):

    run_dir = (
        FINAL_RESULT_ROOT
        / f"hlna1_formal_400ep_seed{seed}"
    )


    required = [
        "metrics.json",
        "gt_labels.npy",
        "pred_labels.npy",
        "pred_concat_z_kmeans.npy",
        "z_concat.npy",
        "z_bsrr.npy",
    ]


    missing = [
        name
        for name in required
        if not (
            run_dir / name
        ).exists()
    ]


    if missing:

        raise RuntimeError(
            f"HLN-A1 seed {seed} "
            f"is incomplete: {missing}"
        )


    with (
        run_dir
        / "metrics.json"
    ).open(
        "r",
        encoding="utf-8",
    ) as f:

        metrics = json.load(f)


    gt = np.load(
        run_dir
        / "gt_labels.npy"
    )

    pred_official = np.load(
        run_dir
        / "pred_labels.npy"
    )

    pred_raw = np.load(
        run_dir
        / "pred_concat_z_kmeans.npy"
    )


    # --------------------------------------------------------
    # Independently recompute official result
    # --------------------------------------------------------

    official_ari = (
        adjusted_rand_score(
            gt,
            pred_official,
        )
    )

    official_nmi = (
        normalized_mutual_info_score(
            gt,
            pred_official,
            average_method="max",
        )
    )


    raw_ari = (
        adjusted_rand_score(
            gt,
            pred_raw,
        )
    )

    raw_nmi = (
        normalized_mutual_info_score(
            gt,
            pred_raw,
            average_method="max",
        )
    )


    # metrics.json must agree exactly
    assert np.isclose(
        official_ari,
        float(metrics["ARI"]),
    )

    assert np.isclose(
        official_nmi,
        float(metrics["NMI"]),
    )


    rows.append(
        {
            "dataset": "HLN-A1",
            "seed": seed,

            "ARI":
                official_ari,

            "NMI":
                official_nmi,

            "raw_ARI":
                raw_ari,

            "raw_NMI":
                raw_nmi,

            "delta_ARI_BSRR":
                official_ari
                - raw_ari,

            "delta_NMI_BSRR":
                official_nmi
                - raw_nmi,
        }
    )


raw_df = pd.DataFrame(
    rows
).sort_values(
    "seed"
).reset_index(
    drop=True
)


# ============================================================
# Formal summary
# std uses ddof=0, consistent with our sensitivity summaries
# ============================================================

summary = {

    "dataset":
        "HLN-A1",

    "n_seeds":
        len(raw_df),

    "epochs":
        400,

    "ARI_mean":
        raw_df[
            "ARI"
        ].mean(),

    "ARI_std":
        raw_df[
            "ARI"
        ].std(
            ddof=0
        ),

    "NMI_mean":
        raw_df[
            "NMI"
        ].mean(),

    "NMI_std":
        raw_df[
            "NMI"
        ].std(
            ddof=0
        ),

    "raw_ARI_mean":
        raw_df[
            "raw_ARI"
        ].mean(),

    "raw_ARI_std":
        raw_df[
            "raw_ARI"
        ].std(
            ddof=0
        ),

    "raw_NMI_mean":
        raw_df[
            "raw_NMI"
        ].mean(),

    "raw_NMI_std":
        raw_df[
            "raw_NMI"
        ].std(
            ddof=0
        ),

    "mean_delta_ARI_BSRR":
        raw_df[
            "delta_ARI_BSRR"
        ].mean(),

    "mean_delta_NMI_BSRR":
        raw_df[
            "delta_NMI_BSRR"
        ].mean(),

    "BSRR_ARI_improved_seeds":
        int(
            (
                raw_df[
                    "delta_ARI_BSRR"
                ] > 0
            ).sum()
        ),

    "BSRR_NMI_improved_seeds":
        int(
            (
                raw_df[
                    "delta_NMI_BSRR"
                ] > 0
            ).sum()
        ),
}


summary_df = pd.DataFrame(
    [summary]
)


# ============================================================
# Save
# ============================================================

RAW_CSV = (
    SUMMARY_DIR
    / "hlna1_10seed_400ep_raw.csv"
)

SUMMARY_CSV = (
    SUMMARY_DIR
    / "hlna1_10seed_400ep_summary.csv"
)


raw_df.to_csv(
    RAW_CSV,
    index=False,
)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
)


# ============================================================
# Print
# ============================================================

print("=" * 90)
print("HLN-A1 | 10 SEEDS | 400 EPOCHS")
print("=" * 90)

print("\nPer-seed results:\n")

print(
    raw_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


print("\n" + "-" * 90)

print(
    "Official SpaMGCL "
    "(concat-Z -> BSRR -> KMeans)"
)

print(
    f"ARI = "
    f"{summary['ARI_mean']:.6f}"
    f" ± "
    f"{summary['ARI_std']:.6f}"
)

print(
    f"NMI = "
    f"{summary['NMI_mean']:.6f}"
    f" ± "
    f"{summary['NMI_std']:.6f}"
)


print("\nRaw concat-Z diagnostic")

print(
    f"ARI = "
    f"{summary['raw_ARI_mean']:.6f}"
    f" ± "
    f"{summary['raw_ARI_std']:.6f}"
)

print(
    f"NMI = "
    f"{summary['raw_NMI_mean']:.6f}"
    f" ± "
    f"{summary['raw_NMI_std']:.6f}"
)


print("\nBSRR mean effect")

print(
    f"Delta ARI = "
    f"{summary['mean_delta_ARI_BSRR']:+.6f}"
)

print(
    f"Delta NMI = "
    f"{summary['mean_delta_NMI_BSRR']:+.6f}"
)

print(
    "ARI improved seeds:",
    f"{summary['BSRR_ARI_improved_seeds']}/10",
)

print(
    "NMI improved seeds:",
    f"{summary['BSRR_NMI_improved_seeds']}/10",
)


print("\nSaved:")

print(RAW_CSV)
print(SUMMARY_CSV)


assert len(
    raw_df
) == 10

assert (
    raw_df["seed"].tolist()
    == list(range(10))
)


print(
    "\nPASS: HLN-A1 10-seed "
    "formal summary complete."
)

HLN-A1 | 10 SEEDS | 400 EPOCHS

Per-seed results:

dataset  seed      ARI      NMI  raw_ARI  raw_NMI  delta_ARI_BSRR  delta_NMI_BSRR
 HLN-A1     0 0.220614 0.358325 0.218924 0.353421        0.001690        0.004904
 HLN-A1     1 0.273804 0.393244 0.281581 0.394487       -0.007777       -0.001244
 HLN-A1     2 0.278258 0.404178 0.281465 0.403045       -0.003207        0.001132
 HLN-A1     3 0.237522 0.353609 0.237527 0.349381       -0.000005        0.004228
 HLN-A1     4 0.263325 0.387647 0.265982 0.389643       -0.002656       -0.001996
 HLN-A1     5 0.220615 0.345039 0.222539 0.346643       -0.001924       -0.001604
 HLN-A1     6 0.296535 0.399609 0.293513 0.400389        0.003022       -0.000779
 HLN-A1     7 0.231655 0.342726 0.220466 0.341683        0.011189        0.001042
 HLN-A1     8 0.238096 0.356490 0.235449 0.354454        0.002647        0.002036
 HLN-A1     9 0.232451 0.347915 0.237678 0.347981       -0.005227       -0.000066

----------------------------------------------

Cell 15：归档 HLN-A1 正式结果

In [15]:
# ============================================================
# Cell 15
# Archive completed HLN-A1 formal experiment
# ============================================================

from pathlib import Path
import json
import subprocess
import zipfile


ARCHIVE_PATH = Path(
    "/kaggle/working/"
    "SpaMGCL_HLNA1_10seeds_400ep_FINAL.zip"
)


MANIFEST_PATH = (
    SUMMARY_DIR
    / "hlna1_10seed_400ep_manifest.json"
)


# ============================================================
# Git commit
# ============================================================

git_commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(
            PROJECT_ROOT.parent
        ),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()


# ============================================================
# Load summary
# ============================================================

summary_record = (
    summary_df.iloc[0]
    .to_dict()
)


manifest = {

    "experiment":
        "SpaMGCL formal benchmark",

    "dataset":
        "HLN-A1",

    "epochs":
        400,

    "seeds":
        list(range(10)),

    "warm_up_epochs":
        10,

    "official_readout":
        "concat-Z -> BSRR -> KMeans",

    "bsrr_spatial_k":
        3,

    "kmeans_n_init":
        20,

    "kmeans_random_state":
        0,

    "nmi_average_method":
        "max",

    "git_commit":
        git_commit,

    "summary":
        {
            key:
                (
                    value.item()
                    if hasattr(
                        value,
                        "item"
                    )
                    else value
                )
            for key, value
            in summary_record.items()
        },
}


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ============================================================
# Files preserved per seed
# ============================================================

KEEP_FILES = [

    "metrics.json",
    "config.yaml",
    "manifest.json",

    "gt_labels.npy",
    "coords.npy",

    "pred_labels.npy",
    "pred_concat_z_kmeans.npy",
    "pred_q_argmax.npy",

    "z_concat.npy",
    "z_bsrr.npy",

    "z_mean.npy",
    "sc_weights.npy",

    "metrics_epoch400.json",
    "checkpoint_epoch400.pt",
]


if ARCHIVE_PATH.exists():

    ARCHIVE_PATH.unlink()


with zipfile.ZipFile(
    ARCHIVE_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    # --------------------------------------------------------
    # 10 runs
    # --------------------------------------------------------

    for seed in range(10):

        run_dir = (
            FINAL_RESULT_ROOT
            / f"hlna1_formal_400ep_seed{seed}"
        )


        assert (
            run_dir
            / "metrics.json"
        ).exists()


        for filename in (
            KEEP_FILES
        ):

            file_path = (
                run_dir
                / filename
            )


            # Some optional diagnostics
            # may not exist.
            if not file_path.exists():
                continue


            archive_name = (
                Path(
                    "results_final_400"
                )
                / run_dir.name
                / filename
            )


            zf.write(
                file_path,
                arcname=str(
                    archive_name
                ),
            )


    # --------------------------------------------------------
    # Original 10 formal configs
    # --------------------------------------------------------

    for seed in range(10):

        cfg_path = (
            FINAL_CONFIG_DIR
            / (
                f"hlna1_"
                f"formal_400ep_"
                f"seed{seed}.yaml"
            )
        )


        assert cfg_path.exists()


        zf.write(
            cfg_path,
            arcname=str(
                Path(
                    "configs/formal_400ep"
                )
                / cfg_path.name
            ),
        )


    # --------------------------------------------------------
    # Formal runner
    # --------------------------------------------------------

    zf.write(
        RUNNER_FORMAL,
        arcname=(
            "experiments/"
            "run_exp_formal_400.py"
        ),
    )


    # --------------------------------------------------------
    # Summaries + manifest
    # --------------------------------------------------------

    for file_path in [
        RAW_CSV,
        SUMMARY_CSV,
        MANIFEST_PATH,
    ]:

        zf.write(
            file_path,
            arcname=str(
                Path(
                    "formal_summary_400"
                )
                / file_path.name
            ),
        )


size_mb = (
    ARCHIVE_PATH.stat().st_size
    / 1024
    / 1024
)


print("=" * 90)
print("HLN-A1 FORMAL ARCHIVE")
print("=" * 90)

print(
    "Archive:",
    ARCHIVE_PATH,
)

print(
    f"Size: {size_mb:.2f} MB"
)

print(
    "Git commit:",
    git_commit,
)


assert ARCHIVE_PATH.exists()
assert ARCHIVE_PATH.stat().st_size > 0


print(
    "\nPASS: HLN-A1 formal "
    "10-seed archive created."
)

HLN-A1 FORMAL ARCHIVE
Archive: /kaggle/working/SpaMGCL_HLNA1_10seeds_400ep_FINAL.zip
Size: 41.98 MB
Git commit: b4abd3de0c590e024a28add6fba4f0f089837112

PASS: HLN-A1 formal 10-seed archive created.


Cell 16：运行 HLN-D1 10 seeds

In [16]:
# ============================================================
# Cell 16
# HLN-D1 | 10 formal seeds | 400 epochs
# ============================================================

from pathlib import Path
import time


D1_SEEDS_TO_RUN = list(range(10))


completed_dirs = []


print("=" * 90)
print("HLN-D1 FORMAL 400-EPOCH RUNS")
print("=" * 90)

print("Seeds to run:", D1_SEEDS_TO_RUN)
print("Total:", len(D1_SEEDS_TO_RUN))


for i, seed in enumerate(
    D1_SEEDS_TO_RUN,
    start=1,
):

    config_path = (
        FINAL_CONFIG_DIR
        / f"d1_formal_400ep_seed{seed}.yaml"
    )


    assert config_path.exists(), (
        f"Missing config: {config_path}"
    )


    print("\n" + "#" * 90)

    print(
        f"HLN-D1 seed {seed}"
        f" | {i}/{len(D1_SEEDS_TO_RUN)}"
    )

    print("#" * 90)


    start_time = time.time()


    output_dir = run_formal_config(
        config_path
    )


    elapsed_min = (
        time.time()
        - start_time
    ) / 60.0


    completed_dirs.append(
        output_dir
    )


    print(
        f"\nSeed {seed} finished "
        f"in {elapsed_min:.2f} min"
    )


print("\n" + "=" * 90)
print("HLN-D1 ALL REQUESTED SEEDS COMPLETED")
print("=" * 90)

print(
    "Completed in this call:",
    len(completed_dirs),
)


for p in completed_dirs:
    print(" -", p.name)

HLN-D1 FORMAL 400-EPOCH RUNS
Seeds to run: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Total: 10

##########################################################################################
HLN-D1 seed 0 | 1/10
##########################################################################################

FORMAL EXPERIMENT
Dataset : HLN-D1
Seed    : 0
Name    : d1_formal_400ep_seed0
Epochs  : 400
Config  : /kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep/d1_formal_400ep_seed0.yaml
Output  : /kaggle/working/SpaMGCL/SpaMGCL/results_final_400/d1_formal_400ep_seed0

Starting fresh formal run.

Running:
/usr/bin/python3 /kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp_formal_400.py --config /kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep/d1_formal_400ep_seed0.yaml



/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-D1 | spots=3359 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3359, 3359), nnz=12744
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.370173 | rec=0.248054 | mgcl=8.122119 | cluster=3.050508 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.870e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.952 | gradC=0.000e+00
epoch 002/400 | total=8.359020 | rec=0.238144 | mgcl=8.120875 | cluster=3.049739 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.930e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.958 | gradC=0.000e+00
epoch 003/400 | total=8.348646 | rec=0.228612 | mgcl=8.120034 | cluster=3.049047 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.583e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.963 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-D1 | spots=3359 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3359, 3359), nnz=12744
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.349872 | rec=0.226992 | mgcl=8.122880 | cluster=3.048808 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.623e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.970 | gradC=0.000e+00
epoch 002/400 | total=8.339814 | rec=0.218547 | mgcl=8.121267 | cluster=3.048234 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.568e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.975 | gradC=0.000e+00
epoch 003/400 | total=8.330775 | rec=0.210469 | mgcl=8.120307 | cluster=3.047729 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.795e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.978 | gradC=0.000e+00
epoch

...

Dataset: HLN-D1 | spots=3359 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3359, 3359), nnz=12744
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.400756 | rec=0.279445 | mgcl=8.121310 | cluster=3.051596 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.611e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.946 | gradC=0.000e+00
epoch 002/400 | total=8.388640 | rec=0.268163 | mgcl=8.120478 | cluster=3.050722 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.620e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.953 | gradC=0.000e+00
epoch 003/400 | total=8.377419 | rec=0.257548 | mgcl=8.119872 | cluster=3.049953 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.990e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.959 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-D1 | spots=3359 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3359, 3359), nnz=12744
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.380973 | rec=0.259627 | mgcl=8.121346 | cluster=3.053863 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.816e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.916 | gradC=0.000e+00
epoch 002/400 | total=8.369314 | rec=0.249122 | mgcl=8.120192 | cluster=3.052666 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.013e-02 | neg_count=11279522 | snf_masked_positions=0 | effC=10.926 | gradC=0.000e+00
epoch 003/400 | total=8.358519 | rec=0.239153 | mgcl=8.119365 | cluster=3.051585 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.161e-02 | neg_count=11279522 | snf_masked_positions=0 | effC=10.936 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-D1 | spots=3359 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3359, 3359), nnz=12744
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.350972 | rec=0.229622 | mgcl=8.121350 | cluster=3.049772 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.657e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.950 | gradC=0.000e+00
epoch 002/400 | total=8.340958 | rec=0.221065 | mgcl=8.119893 | cluster=3.049099 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.784e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.956 | gradC=0.000e+00
epoch 003/400 | total=8.331631 | rec=0.212854 | mgcl=8.118776 | cluster=3.048502 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.388e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.962 | gradC=0.000e+00
epoch

Cell 17：HLN-D1 10-seed 正式汇总

In [17]:
# ============================================================
# Cell 17
# Generic formal 10-seed summary
# + summarize HLN-D1
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


SUMMARY_DIR = (
    PROJECT_ROOT
    / "formal_summary_400"
)

SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def summarize_formal_dataset(
    dataset_label,
    slug,
    n_seeds=10,
):

    rows = []


    for seed in range(
        n_seeds
    ):

        run_dir = (
            FINAL_RESULT_ROOT
            / (
                f"{slug}_"
                f"formal_400ep_"
                f"seed{seed}"
            )
        )


        required = [
            "metrics.json",
            "gt_labels.npy",
            "pred_labels.npy",
            "pred_concat_z_kmeans.npy",
            "z_concat.npy",
            "z_bsrr.npy",
        ]


        missing = [
            name
            for name in required
            if not (
                run_dir
                / name
            ).exists()
        ]


        if missing:

            raise RuntimeError(
                f"{dataset_label} "
                f"seed {seed} "
                f"is incomplete: "
                f"{missing}"
            )


        # ----------------------------------------------------
        # Load
        # ----------------------------------------------------

        with (
            run_dir
            / "metrics.json"
        ).open(
            "r",
            encoding="utf-8",
        ) as f:

            metrics = json.load(f)


        gt = np.load(
            run_dir
            / "gt_labels.npy"
        )


        pred_official = np.load(
            run_dir
            / "pred_labels.npy"
        )


        pred_raw = np.load(
            run_dir
            / "pred_concat_z_kmeans.npy"
        )


        z_concat = np.load(
            run_dir
            / "z_concat.npy"
        )


        z_bsrr = np.load(
            run_dir
            / "z_bsrr.npy"
        )


        # ----------------------------------------------------
        # Basic shape checks
        # ----------------------------------------------------

        assert (
            len(gt)
            == len(pred_official)
            == len(pred_raw)
            == z_concat.shape[0]
            == z_bsrr.shape[0]
        )


        # ----------------------------------------------------
        # Official
        # ----------------------------------------------------

        official_ari = (
            adjusted_rand_score(
                gt,
                pred_official,
            )
        )


        official_nmi = (
            normalized_mutual_info_score(
                gt,
                pred_official,
                average_method="max",
            )
        )


        # ----------------------------------------------------
        # Raw concat-Z diagnostic
        # ----------------------------------------------------

        raw_ari = (
            adjusted_rand_score(
                gt,
                pred_raw,
            )
        )


        raw_nmi = (
            normalized_mutual_info_score(
                gt,
                pred_raw,
                average_method="max",
            )
        )


        # ----------------------------------------------------
        # Must match metrics.json
        # ----------------------------------------------------

        assert np.isclose(
            official_ari,
            float(
                metrics["ARI"]
            ),
        )


        assert np.isclose(
            official_nmi,
            float(
                metrics["NMI"]
            ),
        )


        # ----------------------------------------------------
        # Protocol metadata
        # ----------------------------------------------------

        official_readout = (
            metrics[
                "official_readout"
            ]
        )


        refinement = (
            metrics[
                "refinement"
            ]
        )


        resolved = (
            metrics[
                "resolved_parameters"
            ][
                "clustering"
            ]
        )


        assert (
            official_readout[
                "embedding"
            ]
            == "concat_z"
        )


        assert (
            official_readout[
                "refinement_enabled"
            ]
            is True
        )


        assert (
            official_readout[
                "refinement_method"
            ]
            == "bsrr"
        )


        assert (
            official_readout[
                "clustering_method"
            ]
            == "kmeans"
        )


        assert int(
            refinement[
                "spatial_k"
            ]
        ) == 3


        assert int(
            resolved[
                "n_init"
            ]
        ) == 20


        assert int(
            resolved[
                "random_state"
            ]
        ) == 0


        # ----------------------------------------------------
        # Record
        # ----------------------------------------------------

        rows.append(
            {
                "dataset":
                    dataset_label,

                "seed":
                    seed,

                "ARI":
                    official_ari,

                "NMI":
                    official_nmi,

                "raw_ARI":
                    raw_ari,

                "raw_NMI":
                    raw_nmi,

                "delta_ARI_BSRR":
                    official_ari
                    - raw_ari,

                "delta_NMI_BSRR":
                    official_nmi
                    - raw_nmi,
            }
        )


    # ========================================================
    # DataFrame
    # ========================================================

    raw_df = (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "seed"
        )
        .reset_index(
            drop=True
        )
    )


    assert len(
        raw_df
    ) == n_seeds


    assert (
        raw_df[
            "seed"
        ].tolist()
        ==
        list(
            range(
                n_seeds
            )
        )
    )


    # ========================================================
    # Summary
    # ========================================================

    summary = {

        "dataset":
            dataset_label,

        "n_seeds":
            n_seeds,

        "epochs":
            400,

        "ARI_mean":
            raw_df[
                "ARI"
            ].mean(),

        "ARI_std":
            raw_df[
                "ARI"
            ].std(
                ddof=0
            ),

        "NMI_mean":
            raw_df[
                "NMI"
            ].mean(),

        "NMI_std":
            raw_df[
                "NMI"
            ].std(
                ddof=0
            ),

        "raw_ARI_mean":
            raw_df[
                "raw_ARI"
            ].mean(),

        "raw_ARI_std":
            raw_df[
                "raw_ARI"
            ].std(
                ddof=0
            ),

        "raw_NMI_mean":
            raw_df[
                "raw_NMI"
            ].mean(),

        "raw_NMI_std":
            raw_df[
                "raw_NMI"
            ].std(
                ddof=0
            ),

        "mean_delta_ARI_BSRR":
            raw_df[
                "delta_ARI_BSRR"
            ].mean(),

        "mean_delta_NMI_BSRR":
            raw_df[
                "delta_NMI_BSRR"
            ].mean(),

        "BSRR_ARI_improved_seeds":
            int(
                (
                    raw_df[
                        "delta_ARI_BSRR"
                    ] > 0
                ).sum()
            ),

        "BSRR_NMI_improved_seeds":
            int(
                (
                    raw_df[
                        "delta_NMI_BSRR"
                    ] > 0
                ).sum()
            ),
    }


    summary_df = (
        pd.DataFrame(
            [summary]
        )
    )


    # ========================================================
    # Save
    # ========================================================

    raw_csv = (
        SUMMARY_DIR
        / (
            f"{slug}_"
            f"10seed_400ep_raw.csv"
        )
    )


    summary_csv = (
        SUMMARY_DIR
        / (
            f"{slug}_"
            f"10seed_400ep_summary.csv"
        )
    )


    raw_df.to_csv(
        raw_csv,
        index=False,
    )


    summary_df.to_csv(
        summary_csv,
        index=False,
    )


    # ========================================================
    # Print
    # ========================================================

    print("=" * 90)

    print(
        f"{dataset_label} "
        "| 10 SEEDS "
        "| 400 EPOCHS"
    )

    print("=" * 90)


    print(
        "\nPer-seed results:\n"
    )


    print(
        raw_df.to_string(
            index=False,
            float_format=(
                lambda x:
                f"{x:.6f}"
            ),
        )
    )


    print(
        "\n"
        + "-"
        * 90
    )


    print(
        "Official SpaMGCL "
        "(concat-Z -> BSRR -> KMeans)"
    )


    print(
        f"ARI = "
        f"{summary['ARI_mean']:.6f}"
        f" ± "
        f"{summary['ARI_std']:.6f}"
    )


    print(
        f"NMI = "
        f"{summary['NMI_mean']:.6f}"
        f" ± "
        f"{summary['NMI_std']:.6f}"
    )


    print(
        "\nRaw concat-Z diagnostic"
    )


    print(
        f"ARI = "
        f"{summary['raw_ARI_mean']:.6f}"
        f" ± "
        f"{summary['raw_ARI_std']:.6f}"
    )


    print(
        f"NMI = "
        f"{summary['raw_NMI_mean']:.6f}"
        f" ± "
        f"{summary['raw_NMI_std']:.6f}"
    )


    print(
        "\nBSRR mean effect"
    )


    print(
        f"Delta ARI = "
        f"{summary['mean_delta_ARI_BSRR']:+.6f}"
    )


    print(
        f"Delta NMI = "
        f"{summary['mean_delta_NMI_BSRR']:+.6f}"
    )


    print(
        "ARI improved seeds:",
        (
            f"{summary['BSRR_ARI_improved_seeds']}"
            f"/{n_seeds}"
        ),
    )


    print(
        "NMI improved seeds:",
        (
            f"{summary['BSRR_NMI_improved_seeds']}"
            f"/{n_seeds}"
        ),
    )


    print(
        "\nSaved:"
    )

    print(
        raw_csv
    )

    print(
        summary_csv
    )


    print(
        f"\nPASS: {dataset_label} "
        "formal summary complete."
    )


    return (
        raw_df,
        summary_df,
        raw_csv,
        summary_csv,
    )


# ============================================================
# HLN-D1
# ============================================================

(
    d1_raw_df,
    d1_summary_df,
    d1_raw_csv,
    d1_summary_csv,
) = summarize_formal_dataset(
    dataset_label="HLN-D1",
    slug="d1",
)

HLN-D1 | 10 SEEDS | 400 EPOCHS

Per-seed results:

dataset  seed      ARI      NMI  raw_ARI  raw_NMI  delta_ARI_BSRR  delta_NMI_BSRR
 HLN-D1     0 0.253712 0.338492 0.254722 0.336775       -0.001010        0.001718
 HLN-D1     1 0.241805 0.331744 0.243298 0.334956       -0.001493       -0.003213
 HLN-D1     2 0.268406 0.343868 0.269572 0.341668       -0.001166        0.002200
 HLN-D1     3 0.219876 0.300542 0.218049 0.299634        0.001827        0.000908
 HLN-D1     4 0.263218 0.335151 0.265054 0.338495       -0.001836       -0.003344
 HLN-D1     5 0.240077 0.330739 0.254950 0.337383       -0.014874       -0.006644
 HLN-D1     6 0.230640 0.320952 0.231783 0.320760       -0.001143        0.000192
 HLN-D1     7 0.277148 0.333270 0.259768 0.329254        0.017380        0.004017
 HLN-D1     8 0.241210 0.333543 0.243390 0.338939       -0.002180       -0.005396
 HLN-D1     9 0.234592 0.341347 0.231971 0.333884        0.002621        0.007464

----------------------------------------------

Cell 18：归档 HLN-D1

In [18]:
# ============================================================
# Cell 18
# Generic formal archive function
# + archive HLN-D1
# ============================================================

from pathlib import Path
import json
import subprocess
import zipfile


KEEP_FORMAL_FILES = [

    "metrics.json",
    "config.yaml",
    "manifest.json",

    "gt_labels.npy",
    "coords.npy",

    "pred_labels.npy",
    "pred_concat_z_kmeans.npy",
    "pred_q_argmax.npy",

    "z_concat.npy",
    "z_bsrr.npy",

    "z_mean.npy",
    "sc_weights.npy",

    "metrics_epoch400.json",
    "checkpoint_epoch400.pt",
]


def archive_formal_dataset(
    dataset_label,
    slug,
    summary_df,
    raw_csv,
    summary_csv,
):

    # ========================================================
    # Git commit
    # ========================================================

    git_commit = (
        subprocess.check_output(
            [
                "git",
                "-C",
                str(
                    PROJECT_ROOT.parent
                ),
                "rev-parse",
                "HEAD",
            ],
            text=True,
        )
        .strip()
    )


    # ========================================================
    # Manifest
    # ========================================================

    summary_record = (
        summary_df
        .iloc[0]
        .to_dict()
    )


    manifest = {

        "experiment":
            "SpaMGCL formal benchmark",

        "dataset":
            dataset_label,

        "epochs":
            400,

        "seeds":
            list(
                range(10)
            ),

        "warm_up_epochs":
            10,

        "official_readout":
            (
                "concat-Z -> "
                "BSRR -> KMeans"
            ),

        "bsrr_spatial_k":
            3,

        "kmeans_n_init":
            20,

        "kmeans_random_state":
            0,

        "nmi_average_method":
            "max",

        "git_commit":
            git_commit,

        "summary":
            {
                key:
                    (
                        value.item()
                        if hasattr(
                            value,
                            "item"
                        )
                        else value
                    )

                for key, value
                in summary_record.items()
            },
    }


    manifest_path = (
        SUMMARY_DIR
        / (
            f"{slug}_"
            f"10seed_400ep_"
            f"manifest.json"
        )
    )


    manifest_path.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    # ========================================================
    # ZIP
    # ========================================================

    pretty_name = (
        dataset_label
        .replace("-", "")
        .replace(".", "")
    )


    archive_path = Path(
        "/kaggle/working/"
        f"SpaMGCL_{pretty_name}_"
        f"10seeds_400ep_FINAL.zip"
    )


    if archive_path.exists():

        archive_path.unlink()


    with zipfile.ZipFile(
        archive_path,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as zf:

        # ----------------------------------------------------
        # Result directories
        # ----------------------------------------------------

        for seed in range(10):

            run_dir = (
                FINAL_RESULT_ROOT
                / (
                    f"{slug}_"
                    f"formal_400ep_"
                    f"seed{seed}"
                )
            )


            assert (
                run_dir
                / "metrics.json"
            ).exists()


            for filename in (
                KEEP_FORMAL_FILES
            ):

                file_path = (
                    run_dir
                    / filename
                )


                if not (
                    file_path.exists()
                ):

                    continue


                archive_name = (
                    Path(
                        "results_final_400"
                    )
                    / run_dir.name
                    / filename
                )


                zf.write(
                    file_path,
                    arcname=str(
                        archive_name
                    ),
                )


        # ----------------------------------------------------
        # Original formal YAML
        # ----------------------------------------------------

        for seed in range(10):

            cfg_path = (
                FINAL_CONFIG_DIR
                / (
                    f"{slug}_"
                    f"formal_400ep_"
                    f"seed{seed}.yaml"
                )
            )


            assert (
                cfg_path.exists()
            )


            zf.write(
                cfg_path,
                arcname=str(
                    Path(
                        "configs/"
                        "formal_400ep"
                    )
                    / cfg_path.name
                ),
            )


        # ----------------------------------------------------
        # Formal runner
        # ----------------------------------------------------

        zf.write(
            RUNNER_FORMAL,
            arcname=(
                "experiments/"
                "run_exp_formal_400.py"
            ),
        )


        # ----------------------------------------------------
        # Summary
        # ----------------------------------------------------

        for file_path in [
            raw_csv,
            summary_csv,
            manifest_path,
        ]:

            zf.write(
                file_path,
                arcname=str(
                    Path(
                        "formal_summary_400"
                    )
                    / file_path.name
                ),
            )


    size_mb = (
        archive_path.stat().st_size
        / 1024
        / 1024
    )


    print("=" * 90)

    print(
        f"{dataset_label} "
        "FORMAL ARCHIVE"
    )

    print("=" * 90)


    print(
        "Archive:",
        archive_path,
    )


    print(
        f"Size: "
        f"{size_mb:.2f} MB"
    )


    print(
        "Git commit:",
        git_commit,
    )


    assert (
        archive_path.exists()
    )


    assert (
        archive_path.stat().st_size
        > 0
    )


    print(
        f"\nPASS: {dataset_label} "
        "formal 10-seed archive created."
    )


    return archive_path


# ============================================================
# Archive HLN-D1
# ============================================================

D1_ARCHIVE = (
    archive_formal_dataset(
        dataset_label="HLN-D1",
        slug="d1",
        summary_df=d1_summary_df,
        raw_csv=d1_raw_csv,
        summary_csv=d1_summary_csv,
    )
)

HLN-D1 FORMAL ARCHIVE
Archive: /kaggle/working/SpaMGCL_HLND1_10seeds_400ep_FINAL.zip
Size: 40.70 MB
Git commit: b4abd3de0c590e024a28add6fba4f0f089837112

PASS: HLN-D1 formal 10-seed archive created.


Cell 19：运行 E18.5 × 10 seeds × 400 epochs

In [19]:
# ============================================================
# Cell 19
# E18.5 | 10 formal seeds | 400 epochs
# ============================================================

import time


E185_SEEDS_TO_RUN = list(range(10))

completed_dirs = []


print("=" * 90)
print("E18.5 FORMAL 400-EPOCH RUNS")
print("=" * 90)

print("Seeds to run:", E185_SEEDS_TO_RUN)
print("Total:", len(E185_SEEDS_TO_RUN))


for i, seed in enumerate(
    E185_SEEDS_TO_RUN,
    start=1,
):

    config_path = (
        FINAL_CONFIG_DIR
        / f"e185_formal_400ep_seed{seed}.yaml"
    )

    assert config_path.exists(), (
        f"Missing config: {config_path}"
    )


    print("\n" + "#" * 90)

    print(
        f"E18.5 seed {seed}"
        f" | {i}/{len(E185_SEEDS_TO_RUN)}"
    )

    print("#" * 90)


    start_time = time.time()


    output_dir = run_formal_config(
        config_path
    )


    elapsed_min = (
        time.time()
        - start_time
    ) / 60.0


    completed_dirs.append(
        output_dir
    )


    print(
        f"\nSeed {seed} finished "
        f"in {elapsed_min:.2f} min"
    )


print("\n" + "=" * 90)
print("E18.5 ALL REQUESTED SEEDS COMPLETED")
print("=" * 90)

print(
    "Completed in this call:",
    len(completed_dirs),
)

for p in completed_dirs:
    print(" -", p.name)

E18.5 FORMAL 400-EPOCH RUNS
Seeds to run: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Total: 10

##########################################################################################
E18.5 seed 0 | 1/10
##########################################################################################

FORMAL EXPERIMENT
Dataset : E18.5
Seed    : 0
Name    : e185_formal_400ep_seed0
Epochs  : 400
Config  : /kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep/e185_formal_400ep_seed0.yaml
Output  : /kaggle/working/SpaMGCL/SpaMGCL/results_final_400/e185_formal_400ep_seed0

Starting fresh formal run.

Running:
/usr/bin/python3 /kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp_formal_400.py --config /kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep/e185_formal_400ep_seed0.yaml

Dataset: E18.5 | spots=2129 | device=cuda
Modalities: RNA, ATAC | label=Combined_Clusters
Spatial graph: shape=(2129, 2129), nnz=7784
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=23.166977 | rec=0

Cell 20：E18.5 正式 10-seed 汇总

In [20]:
# ============================================================
# Cell 20
# E18.5 10-seed formal summary
# ============================================================

(
    e185_raw_df,
    e185_summary_df,
    e185_raw_csv,
    e185_summary_csv,
) = summarize_formal_dataset(
    dataset_label="E18.5",
    slug="e185",
)

E18.5 | 10 SEEDS | 400 EPOCHS

Per-seed results:

dataset  seed      ARI      NMI  raw_ARI  raw_NMI  delta_ARI_BSRR  delta_NMI_BSRR
  E18.5     0 0.437536 0.554442 0.401101 0.548727        0.036435        0.005715
  E18.5     1 0.423785 0.547109 0.406558 0.545121        0.017227        0.001987
  E18.5     2 0.466861 0.572917 0.472311 0.575519       -0.005450       -0.002601
  E18.5     3 0.331331 0.545151 0.299827 0.515287        0.031504        0.029865
  E18.5     4 0.512318 0.575264 0.460753 0.570913        0.051565        0.004351
  E18.5     5 0.451471 0.557234 0.401414 0.530304        0.050057        0.026930
  E18.5     6 0.469072 0.573669 0.454286 0.570357        0.014786        0.003312
  E18.5     7 0.342017 0.520771 0.456730 0.560400       -0.114713       -0.039629
  E18.5     8 0.335712 0.564626 0.362809 0.550310       -0.027096        0.014317
  E18.5     9 0.505216 0.571096 0.421147 0.551283        0.084070        0.019813

-----------------------------------------------

Cell 21：归档 E18.5

In [21]:
# ============================================================
# Cell 21
# Archive E18.5 formal results
# ============================================================

E185_ARCHIVE = (
    archive_formal_dataset(
        dataset_label="E18.5",
        slug="e185",
        summary_df=e185_summary_df,
        raw_csv=e185_raw_csv,
        summary_csv=e185_summary_csv,
    )
)

E18.5 FORMAL ARCHIVE
Archive: /kaggle/working/SpaMGCL_E185_10seeds_400ep_FINAL.zip
Size: 125.70 MB
Git commit: b4abd3de0c590e024a28add6fba4f0f089837112

PASS: E18.5 formal 10-seed archive created.


Cell 22：S2-E15 × 10 seeds × 400 epochs

In [22]:
# ============================================================
# Cell 22
# S2-E15 | 10 formal seeds | 400 epochs
# ============================================================

import time


S2E15_SEEDS_TO_RUN = list(range(10))

completed_dirs = []


print("=" * 90)
print("S2-E15 FORMAL 400-EPOCH RUNS")
print("=" * 90)

print("Seeds to run:", S2E15_SEEDS_TO_RUN)
print("Total:", len(S2E15_SEEDS_TO_RUN))


for i, seed in enumerate(
    S2E15_SEEDS_TO_RUN,
    start=1,
):

    config_path = (
        FINAL_CONFIG_DIR
        / f"s2e15_formal_400ep_seed{seed}.yaml"
    )

    assert config_path.exists(), (
        f"Missing config: {config_path}"
    )


    print("\n" + "#" * 90)

    print(
        f"S2-E15 seed {seed}"
        f" | {i}/{len(S2E15_SEEDS_TO_RUN)}"
    )

    print("#" * 90)


    start_time = time.time()


    output_dir = run_formal_config(
        config_path
    )


    elapsed_min = (
        time.time()
        - start_time
    ) / 60.0


    completed_dirs.append(
        output_dir
    )


    print(
        f"\nSeed {seed} finished "
        f"in {elapsed_min:.2f} min"
    )


print("\n" + "=" * 90)
print("S2-E15 ALL REQUESTED SEEDS COMPLETED")
print("=" * 90)

print(
    "Completed in this call:",
    len(completed_dirs),
)

for p in completed_dirs:
    print(" -", p.name)

S2-E15 FORMAL 400-EPOCH RUNS
Seeds to run: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Total: 10

##########################################################################################
S2-E15 seed 0 | 1/10
##########################################################################################

FORMAL EXPERIMENT
Dataset : S2-E15
Seed    : 0
Name    : s2e15_formal_400ep_seed0
Epochs  : 400
Config  : /kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep/s2e15_formal_400ep_seed0.yaml
Output  : /kaggle/working/SpaMGCL/SpaMGCL/results_final_400/s2e15_formal_400ep_seed0

Starting fresh formal run.

Running:
/usr/bin/python3 /kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp_formal_400.py --config /kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep/s2e15_formal_400ep_seed0.yaml



/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E15 | spots=1939 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(1939, 1939), nnz=7062
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=22.891996 | rec=0.165808 | mgcl=7.575396 | cluster=3.369661 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.919e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.972 | gradC=0.000e+00
epoch 002/400 | total=22.847919 | rec=0.153040 | mgcl=7.564960 | cluster=3.369064 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.980e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.979 | gradC=0.000e+00
epoch 003/400 | total=22.801908 | rec=0.141745 | mgcl=7.553388 | cluster=3.368593 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.761e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.985 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E15 | spots=1939 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(1939, 1939), nnz=7062
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=22.889145 | rec=0.161898 | mgcl=7.575749 | cluster=3.369750 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.955e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.974 | gradC=0.000e+00
epoch 002/400 | total=22.842104 | rec=0.149705 | mgcl=7.564133 | cluster=3.369132 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.816e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.980 | gradC=0.000e+00
epoch 003/400 | total=22.789713 | rec=0.138926 | mgcl=7.550262 | cluster=3.368660 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.825e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.985 | gradC=0.000e+00
epoch

...

Dataset: S2-E15 | spots=1939 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(1939, 1939), nnz=7062
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=22.894012 | rec=0.162945 | mgcl=7.577023 | cluster=3.368985 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.110e-02 | neg_count=3757782 | snf_masked_positions=0 | effC=14.979 | gradC=0.000e+00
epoch 002/400 | total=22.857248 | rec=0.150657 | mgcl=7.568863 | cluster=3.368510 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.404e-02 | neg_count=3757782 | snf_masked_positions=0 | effC=14.985 | gradC=0.000e+00
epoch 003/400 | total=22.830036 | rec=0.139443 | mgcl=7.563531 | cluster=3.368177 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.405e-02 | neg_count=3757782 | snf_masked_positions=0 | effC=14.990 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E15 | spots=1939 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(1939, 1939), nnz=7062
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=22.896702 | rec=0.163825 | mgcl=7.577625 | cluster=3.369317 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.180e-02 | neg_count=3757782 | snf_masked_positions=0 | effC=14.976 | gradC=0.000e+00
epoch 002/400 | total=22.856598 | rec=0.150955 | mgcl=7.568548 | cluster=3.368789 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.402e-02 | neg_count=3757782 | snf_masked_positions=0 | effC=14.983 | gradC=0.000e+00
epoch 003/400 | total=22.829460 | rec=0.139290 | mgcl=7.563390 | cluster=3.368339 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.329e-02 | neg_count=3757782 | snf_masked_positions=0 | effC=14.988 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E15 | spots=1939 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(1939, 1939), nnz=7062
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=22.890804 | rec=0.164342 | mgcl=7.575487 | cluster=3.368723 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.708e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.983 | gradC=0.000e+00
epoch 002/400 | total=22.857164 | rec=0.151799 | mgcl=7.568455 | cluster=3.368379 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.938e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.987 | gradC=0.000e+00
epoch 003/400 | total=22.831722 | rec=0.140405 | mgcl=7.563772 | cluster=3.368101 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.173e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.990 | gradC=0.000e+00
epoch

Cell 23：S2-E15 10-seed 汇总

In [25]:
# ============================================================
# Cell 23
# S2-E15 10-seed formal summary
# ============================================================

(
    s2e15_raw_df,
    s2e15_summary_df,
    s2e15_raw_csv,
    s2e15_summary_csv,
) = summarize_formal_dataset(
    dataset_label="S2-E15",
    slug="s2e15",
)

S2-E15 | 10 SEEDS | 400 EPOCHS

Per-seed results:

dataset  seed      ARI      NMI  raw_ARI  raw_NMI  delta_ARI_BSRR  delta_NMI_BSRR
 S2-E15     0 0.354097 0.559494 0.334241 0.547586        0.019856        0.011908
 S2-E15     1 0.350598 0.565109 0.335657 0.544157        0.014941        0.020952
 S2-E15     2 0.402815 0.565286 0.349274 0.564561        0.053540        0.000725
 S2-E15     3 0.355195 0.559479 0.337200 0.546592        0.017994        0.012887
 S2-E15     4 0.348234 0.557973 0.308805 0.559765        0.039429       -0.001792
 S2-E15     5 0.435103 0.587211 0.367019 0.554885        0.068085        0.032325
 S2-E15     6 0.382066 0.572218 0.356319 0.551420        0.025748        0.020798
 S2-E15     7 0.371216 0.544833 0.352778 0.547767        0.018438       -0.002934
 S2-E15     8 0.443923 0.598365 0.421653 0.584146        0.022270        0.014219
 S2-E15     9 0.374399 0.572446 0.349875 0.534780        0.024523        0.037666

----------------------------------------------

Cell 24：归档 S2-E15

In [28]:
# ============================================================
# Cell 24
# Archive S2-E15 formal results
# ============================================================

S2E15_ARCHIVE = (
    archive_formal_dataset(
        dataset_label="S2-E15",
        slug="s2e15",
        summary_df=s2e15_summary_df,
        raw_csv=s2e15_raw_csv,
        summary_csv=s2e15_summary_csv,
    )
)

S2-E15 FORMAL ARCHIVE
Archive: /kaggle/working/SpaMGCL_S2E15_10seeds_400ep_FINAL.zip
Size: 117.99 MB
Git commit: b4abd3de0c590e024a28add6fba4f0f089837112

PASS: S2-E15 formal 10-seed archive created.


Cell 25：S2-E18 × 10 seeds × 400 epochs

In [29]:
# ============================================================
# Cell 25
# S2-E18 | 10 formal seeds | 400 epochs
# ============================================================

import time


S2E18_SEEDS_TO_RUN = list(range(10))

completed_dirs = []


print("=" * 90)
print("S2-E18 FORMAL 400-EPOCH RUNS")
print("=" * 90)

print("Seeds to run:", S2E18_SEEDS_TO_RUN)
print("Total:", len(S2E18_SEEDS_TO_RUN))


for i, seed in enumerate(
    S2E18_SEEDS_TO_RUN,
    start=1,
):

    config_path = (
        FINAL_CONFIG_DIR
        / f"s2e18_formal_400ep_seed{seed}.yaml"
    )

    assert config_path.exists(), (
        f"Missing config: {config_path}"
    )


    print("\n" + "#" * 90)

    print(
        f"S2-E18 seed {seed}"
        f" | {i}/{len(S2E18_SEEDS_TO_RUN)}"
    )

    print("#" * 90)


    start_time = time.time()


    output_dir = run_formal_config(
        config_path
    )


    elapsed_min = (
        time.time()
        - start_time
    ) / 60.0


    completed_dirs.append(
        output_dir
    )


    print(
        f"\nSeed {seed} finished "
        f"in {elapsed_min:.2f} min"
    )


print("\n" + "=" * 90)
print("S2-E18 ALL REQUESTED SEEDS COMPLETED")
print("=" * 90)

print(
    "Completed in this call:",
    len(completed_dirs),
)

for p in completed_dirs:
    print(" -", p.name)

S2-E18 FORMAL 400-EPOCH RUNS
Seeds to run: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Total: 10

##########################################################################################
S2-E18 seed 0 | 1/10
##########################################################################################

FORMAL EXPERIMENT
Dataset : S2-E18
Seed    : 0
Name    : s2e18_formal_400ep_seed0
Epochs  : 400
Config  : /kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep/s2e18_formal_400ep_seed0.yaml
Output  : /kaggle/working/SpaMGCL/SpaMGCL/results_final_400/s2e18_formal_400ep_seed0

Starting fresh formal run.

Running:
/usr/bin/python3 /kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp_formal_400.py --config /kaggle/working/SpaMGCL/SpaMGCL/configs/formal_400ep/s2e18_formal_400ep_seed0.yaml



/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E18 | spots=2248 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(2248, 2248), nnz=8286
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=23.328171 | rec=0.160767 | mgcl=7.722468 | cluster=3.435536 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.855e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.982 | gradC=0.000e+00
epoch 002/400 | total=23.283112 | rec=0.148299 | mgcl=7.711605 | cluster=3.435190 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.357e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.987 | gradC=0.000e+00
epoch 003/400 | total=23.231668 | rec=0.137329 | mgcl=7.698113 | cluster=3.434907 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.303e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.990 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E18 | spots=2248 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(2248, 2248), nnz=8286
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=23.324621 | rec=0.155365 | mgcl=7.723085 | cluster=3.436096 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.948e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.978 | gradC=0.000e+00
epoch 002/400 | total=23.275263 | rec=0.143727 | mgcl=7.710512 | cluster=3.435551 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.556e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.983 | gradC=0.000e+00
epoch 003/400 | total=23.212105 | rec=0.133637 | mgcl=7.692823 | cluster=3.435139 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.940e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.988 | gradC=0.000e+00
epoch

...

Dataset: S2-E18 | spots=2248 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(2248, 2248), nnz=8286
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=23.338100 | rec=0.162293 | mgcl=7.725269 | cluster=3.435690 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.105e-02 | neg_count=5051256 | snf_masked_positions=0 | effC=15.978 | gradC=0.000e+00
epoch 002/400 | total=23.296076 | rec=0.150283 | mgcl=7.715264 | cluster=3.435288 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.443e-02 | neg_count=5051256 | snf_masked_positions=0 | effC=15.984 | gradC=0.000e+00
epoch 003/400 | total=23.260891 | rec=0.139457 | mgcl=7.707144 | cluster=3.434963 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.651e-02 | neg_count=5051256 | snf_masked_positions=0 | effC=15.988 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E18 | spots=2248 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(2248, 2248), nnz=8286
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=23.333382 | rec=0.160937 | mgcl=7.724148 | cluster=3.435792 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.161e-02 | neg_count=5051256 | snf_masked_positions=0 | effC=15.978 | gradC=0.000e+00
epoch 002/400 | total=23.293091 | rec=0.148142 | mgcl=7.714983 | cluster=3.435352 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.496e-02 | neg_count=5051256 | snf_masked_positions=0 | effC=15.984 | gradC=0.000e+00
epoch 003/400 | total=23.258495 | rec=0.136740 | mgcl=7.707252 | cluster=3.434995 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.642e-02 | neg_count=5051256 | snf_masked_positions=0 | effC=15.988 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E18 | spots=2248 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(2248, 2248), nnz=8286
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=23.329287 | rec=0.159029 | mgcl=7.723419 | cluster=3.435204 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.860e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.986 | gradC=0.000e+00
epoch 002/400 | total=23.296541 | rec=0.146848 | mgcl=7.716564 | cluster=3.434880 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.225e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.990 | gradC=0.000e+00
epoch 003/400 | total=23.272547 | rec=0.135765 | mgcl=7.712261 | cluster=3.434608 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.635e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.993 | gradC=0.000e+00
epoch

Cell 26：S2-E18 10-seed 汇总

In [30]:
# ============================================================
# Cell 26
# S2-E18 10-seed formal summary
# ============================================================

(
    s2e18_raw_df,
    s2e18_summary_df,
    s2e18_raw_csv,
    s2e18_summary_csv,
) = summarize_formal_dataset(
    dataset_label="S2-E18",
    slug="s2e18",
)

S2-E18 | 10 SEEDS | 400 EPOCHS

Per-seed results:

dataset  seed      ARI      NMI  raw_ARI  raw_NMI  delta_ARI_BSRR  delta_NMI_BSRR
 S2-E18     0 0.336230 0.496834 0.290137 0.467795        0.046093        0.029040
 S2-E18     1 0.409145 0.523774 0.385519 0.505960        0.023625        0.017815
 S2-E18     2 0.373223 0.487431 0.373322 0.488935       -0.000099       -0.001504
 S2-E18     3 0.427908 0.499067 0.344797 0.506457        0.083111       -0.007390
 S2-E18     4 0.351880 0.487882 0.308638 0.499869        0.043242       -0.011987
 S2-E18     5 0.372266 0.503071 0.356847 0.520494        0.015418       -0.017423
 S2-E18     6 0.302365 0.495802 0.306285 0.497940       -0.003921       -0.002138
 S2-E18     7 0.317121 0.476683 0.303615 0.476731        0.013506       -0.000048
 S2-E18     8 0.353899 0.533084 0.311705 0.504633        0.042194        0.028451
 S2-E18     9 0.340895 0.518709 0.329495 0.504767        0.011400        0.013942

----------------------------------------------

Cell 27：归档 S2-E18

In [31]:
# ============================================================
# Cell 27
# Archive S2-E18 formal results
# ============================================================

S2E18_ARCHIVE = (
    archive_formal_dataset(
        dataset_label="S2-E18",
        slug="s2e18",
        summary_df=s2e18_summary_df,
        raw_csv=s2e18_raw_csv,
        summary_csv=s2e18_summary_csv,
    )
)

S2-E18 FORMAL ARCHIVE
Archive: /kaggle/working/SpaMGCL_S2E18_10seeds_400ep_FINAL.zip
Size: 130.68 MB
Git commit: b4abd3de0c590e024a28add6fba4f0f089837112

PASS: S2-E18 formal 10-seed archive created.


Cell 28：5 个数据集正式 10-seed 总表

In [32]:
# ============================================================
# Cell 28
# Final 5-dataset formal summary
# ============================================================

import pandas as pd
from pathlib import Path


summary_files = [
    SUMMARY_DIR / "hlna1_10seed_400ep_summary.csv",
    SUMMARY_DIR / "d1_10seed_400ep_summary.csv",
    SUMMARY_DIR / "e185_10seed_400ep_summary.csv",
    SUMMARY_DIR / "s2e15_10seed_400ep_summary.csv",
    SUMMARY_DIR / "s2e18_10seed_400ep_summary.csv",
]


for path in summary_files:
    assert path.exists(), f"Missing summary: {path}"


final_summary_df = pd.concat(
    [
        pd.read_csv(path)
        for path in summary_files
    ],
    ignore_index=True,
)


FINAL_5DATASET_CSV = (
    SUMMARY_DIR
    / "SpaMGCL_5datasets_10seeds_400ep_summary.csv"
)


final_summary_df.to_csv(
    FINAL_5DATASET_CSV,
    index=False,
)


display_cols = [
    "dataset",
    "ARI_mean",
    "ARI_std",
    "NMI_mean",
    "NMI_std",
    "raw_ARI_mean",
    "raw_NMI_mean",
    "mean_delta_ARI_BSRR",
    "mean_delta_NMI_BSRR",
    "BSRR_ARI_improved_seeds",
    "BSRR_NMI_improved_seeds",
]


print("=" * 120)
print("SpaMGCL FINAL FORMAL BENCHMARK")
print("5 DATASETS × 10 SEEDS × 400 EPOCHS")
print("=" * 120)

print(
    final_summary_df[
        display_cols
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


print("\n" + "=" * 120)
print("PAPER TABLE FORMAT")
print("=" * 120)


for _, row in final_summary_df.iterrows():

    print(
        f"{row['dataset']:8s} | "
        f"ARI = "
        f"{row['ARI_mean']:.6f} ± "
        f"{row['ARI_std']:.6f} | "
        f"NMI = "
        f"{row['NMI_mean']:.6f} ± "
        f"{row['NMI_std']:.6f}"
    )


macro_ari = final_summary_df[
    "ARI_mean"
].mean()

macro_nmi = final_summary_df[
    "NMI_mean"
].mean()


print("\nMacro-average over 5 datasets")

print(
    f"ARI = {macro_ari:.6f}"
)

print(
    f"NMI = {macro_nmi:.6f}"
)


print(
    "\nSaved:"
)

print(
    FINAL_5DATASET_CSV
)


print(
    "\nPASS: final 5-dataset "
    "formal benchmark summary created."
)

SpaMGCL FINAL FORMAL BENCHMARK
5 DATASETS × 10 SEEDS × 400 EPOCHS
dataset  ARI_mean  ARI_std  NMI_mean  NMI_std  raw_ARI_mean  raw_NMI_mean  mean_delta_ARI_BSRR  mean_delta_NMI_BSRR  BSRR_ARI_improved_seeds  BSRR_NMI_improved_seeds
 HLN-A1  0.249288 0.025243  0.368878 0.023078      0.249512      0.368113            -0.000225             0.000765                        4                        5
 HLN-D1  0.247068 0.017149  0.330965 0.011777      0.247256      0.331175            -0.000187            -0.000210                        3                        6
  E18.5  0.427532 0.064947  0.558228 0.016322      0.413694      0.551822             0.013838             0.006406                        7                        8
 S2-E15  0.381765 0.032992  0.568241 0.014581      0.351282      0.553566             0.030482             0.014675                       10                        8
 S2-E18  0.358493 0.036774  0.502234 0.016871      0.331036      0.497358             0.027457          

Cell 29：50 个正式 run 全局独立核验

In [36]:
# ============================================================
# Cell 29
# Robust global verification of all 50 formal runs
# ============================================================

from pathlib import Path
import json
import yaml
import traceback
import numpy as np
import pandas as pd

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


DATASET_SPECS = {
    "HLN-A1": {
        "slug": "hlna1",
        "spots": 3484,
        "clusters": 10,
    },
    "HLN-D1": {
        "slug": "d1",
        "spots": 3359,
        "clusters": 11,
    },
    "E18.5": {
        "slug": "e185",
        "spots": 2129,
        "clusters": 14,
    },
    "S2-E15": {
        "slug": "s2e15",
        "spots": 1939,
        "clusters": 15,
    },
    "S2-E18": {
        "slug": "s2e18",
        "spots": 2248,
        "clusters": 16,
    },
}


GLOBAL_REQUIRED_FILES = [
    "metrics.json",
    "config.yaml",
    "manifest.json",
    "gt_labels.npy",
    "coords.npy",
    "pred_labels.npy",
    "pred_concat_z_kmeans.npy",
    "pred_q_argmax.npy",
    "z_concat.npy",
    "z_bsrr.npy",
    "metrics_epoch400.json",
    "checkpoint_epoch400.pt",
]


verification_rows = []
failures = []


print("=" * 110)
print("GLOBAL AUDIT: 5 DATASETS × 10 SEEDS × 400 EPOCHS")
print("=" * 110)


for dataset_name, spec in DATASET_SPECS.items():

    slug = spec["slug"]

    print(f"\n[{dataset_name}]")

    for seed in range(10):

        run_dir = (
            FINAL_RESULT_ROOT
            / f"{slug}_formal_400ep_seed{seed}"
        )

        try:

            # ====================================================
            # Directory
            # ====================================================

            if not run_dir.exists():
                raise FileNotFoundError(
                    f"Missing run directory: {run_dir}"
                )


            # ====================================================
            # Required files
            # ====================================================

            missing = [
                name
                for name in GLOBAL_REQUIRED_FILES
                if not (run_dir / name).exists()
            ]

            if missing:
                raise RuntimeError(
                    f"Missing files: {missing}"
                )


            # ====================================================
            # Load config
            # ====================================================

            with (
                run_dir / "config.yaml"
            ).open(
                "r",
                encoding="utf-8",
            ) as f:

                cfg = yaml.safe_load(f)


            # ====================================================
            # Protocol checks
            # ====================================================

            if cfg["experiment"]["dataset"] != dataset_name:
                raise RuntimeError(
                    "Wrong dataset in config"
                )

            if int(cfg["experiment"]["seed"]) != seed:
                raise RuntimeError(
                    "Wrong seed in config"
                )

            if int(cfg["training"]["epochs"]) != 400:
                raise RuntimeError(
                    "epochs != 400"
                )

            if int(cfg["training"]["warm_up_epochs"]) != 10:
                raise RuntimeError(
                    "warm_up_epochs != 10"
                )

            if cfg["clustering"]["embedding"] != "concat_z":
                raise RuntimeError(
                    "embedding != concat_z"
                )

            if cfg["clustering"]["method"] != "kmeans":
                raise RuntimeError(
                    "clustering method != kmeans"
                )

            if int(cfg["clustering"]["n_clusters"]) != spec["clusters"]:
                raise RuntimeError(
                    "wrong number of clusters"
                )

            if int(cfg["clustering"]["n_init"]) != 20:
                raise RuntimeError(
                    "n_init != 20"
                )

            if int(cfg["clustering"]["random_state"]) != 0:
                raise RuntimeError(
                    "KMeans random_state != 0"
                )

            if cfg["refinement"]["enabled"] is not True:
                raise RuntimeError(
                    "BSRR disabled"
                )

            if cfg["refinement"]["method"] != "bsrr":
                raise RuntimeError(
                    "refinement method != bsrr"
                )

            if int(cfg["refinement"]["spatial_k"]) != 3:
                raise RuntimeError(
                    "BSRR spatial_k != 3"
                )

            if cfg["evaluation"]["nmi_average_method"] != "max":
                raise RuntimeError(
                    "NMI average_method != max"
                )


            # ====================================================
            # Load metrics
            # ====================================================

            with (
                run_dir / "metrics.json"
            ).open(
                "r",
                encoding="utf-8",
            ) as f:

                metrics = json.load(f)


            with (
                run_dir / "metrics_epoch400.json"
            ).open(
                "r",
                encoding="utf-8",
            ) as f:

                metrics400 = json.load(f)


            # ====================================================
            # Load arrays
            # ====================================================

            gt = np.load(
                run_dir / "gt_labels.npy"
            )

            pred = np.load(
                run_dir / "pred_labels.npy"
            )

            pred_raw = np.load(
                run_dir / "pred_concat_z_kmeans.npy"
            )

            coords = np.load(
                run_dir / "coords.npy"
            )

            z_concat = np.load(
                run_dir / "z_concat.npy"
            )

            z_bsrr = np.load(
                run_dir / "z_bsrr.npy"
            )


            # ====================================================
            # Shape checks
            # ====================================================

            n = spec["spots"]

            if len(gt) != n:
                raise RuntimeError(
                    f"gt spots mismatch: {len(gt)} != {n}"
                )

            if len(pred) != n:
                raise RuntimeError(
                    "pred length mismatch"
                )

            if len(pred_raw) != n:
                raise RuntimeError(
                    "raw pred length mismatch"
                )

            if coords.shape[0] != n:
                raise RuntimeError(
                    "coords length mismatch"
                )

            if z_concat.shape[0] != n:
                raise RuntimeError(
                    "z_concat length mismatch"
                )

            if z_bsrr.shape[0] != n:
                raise RuntimeError(
                    "z_bsrr length mismatch"
                )

            if z_concat.shape != z_bsrr.shape:
                raise RuntimeError(
                    "z_concat/z_bsrr shape mismatch"
                )


            # ====================================================
            # Independent metrics
            # ====================================================

            ari = adjusted_rand_score(
                gt,
                pred,
            )

            nmi = normalized_mutual_info_score(
                gt,
                pred,
                average_method="max",
            )

            raw_ari = adjusted_rand_score(
                gt,
                pred_raw,
            )

            raw_nmi = normalized_mutual_info_score(
                gt,
                pred_raw,
                average_method="max",
            )


            # ====================================================
            # Cross-check final metrics
            # ====================================================

            if not np.isclose(
                ari,
                float(metrics["ARI"]),
                atol=1e-12,
            ):
                raise RuntimeError(
                    "metrics.json ARI mismatch"
                )

            if not np.isclose(
                nmi,
                float(metrics["NMI"]),
                atol=1e-12,
            ):
                raise RuntimeError(
                    "metrics.json NMI mismatch"
                )

            if not np.isclose(
                ari,
                float(metrics400["ARI"]),
                atol=1e-12,
            ):
                raise RuntimeError(
                    "metrics_epoch400 ARI mismatch"
                )

            if not np.isclose(
                nmi,
                float(metrics400["NMI"]),
                atol=1e-12,
            ):
                raise RuntimeError(
                    "metrics_epoch400 NMI mismatch"
                )


            # ====================================================
            # Official metadata
            # ====================================================

            official = metrics["official_readout"]
            refinement = metrics["refinement"]

            resolved_cluster = (
                metrics[
                    "resolved_parameters"
                ]["clustering"]
            )


            if official["embedding"] != "concat_z":
                raise RuntimeError(
                    "official embedding mismatch"
                )

            if official["refinement_enabled"] is not True:
                raise RuntimeError(
                    "official refinement disabled"
                )

            if official["refinement_method"] != "bsrr":
                raise RuntimeError(
                    "official refinement != bsrr"
                )

            if official["clustering_method"] != "kmeans":
                raise RuntimeError(
                    "official clustering != kmeans"
                )

            if refinement["enabled"] is not True:
                raise RuntimeError(
                    "metrics refinement disabled"
                )

            if refinement["method"] != "bsrr":
                raise RuntimeError(
                    "metrics refinement != bsrr"
                )

            if int(refinement["spatial_k"]) != 3:
                raise RuntimeError(
                    "metrics spatial_k != 3"
                )

            if int(resolved_cluster["n_init"]) != 20:
                raise RuntimeError(
                    "resolved n_init != 20"
                )

            if int(resolved_cluster["random_state"]) != 0:
                raise RuntimeError(
                    "resolved KMeans random_state != 0"
                )


            # ====================================================
            # Loss history
            # ====================================================

            history = metrics.get(
                "loss_history",
                []
            )

            if len(history) != 400:
                raise RuntimeError(
                    f"loss_history length "
                    f"{len(history)} != 400"
                )

            if int(history[-1]["epoch"]) != 400:
                raise RuntimeError(
                    "last loss-history epoch != 400"
                )


            # ====================================================
            # Verified
            # ====================================================

            verification_rows.append(
                {
                    "dataset":
                        dataset_name,

                    "slug":
                        slug,

                    "seed":
                        seed,

                    "epochs":
                        400,

                    "spots":
                        n,

                    "embedding_dim":
                        int(
                            z_concat.shape[1]
                        ),

                    "ARI":
                        ari,

                    "NMI":
                        nmi,

                    "raw_ARI":
                        raw_ari,

                    "raw_NMI":
                        raw_nmi,

                    "delta_ARI_BSRR":
                        ari - raw_ari,

                    "delta_NMI_BSRR":
                        nmi - raw_nmi,

                    "audit":
                        "PASS",
                }
            )


            print(
                f"  PASS seed {seed} | "
                f"ARI={ari:.6f} | "
                f"NMI={nmi:.6f}"
            )


        except Exception as e:

            failure = {
                "dataset":
                    dataset_name,

                "seed":
                    seed,

                "run_dir":
                    str(run_dir),

                "error":
                    repr(e),
            }

            failures.append(
                failure
            )


            print(
                f"  FAIL seed {seed} | "
                f"{type(e).__name__}: {e}"
            )


# ============================================================
# Summary
# ============================================================

print("\n" + "=" * 110)
print("GLOBAL AUDIT SUMMARY")
print("=" * 110)

print(
    "Verified runs:",
    len(verification_rows),
    "/ 50",
)

print(
    "Failed runs:",
    len(failures),
)


if failures:

    print("\nFailures:")

    for failure in failures:

        print(
            f" - "
            f"{failure['dataset']} "
            f"seed {failure['seed']} | "
            f"{failure['error']}"
        )

    raise RuntimeError(
        "GLOBAL AUDIT FAILED. "
        "Fix failures before continuing."
    )


# ============================================================
# Build dataframe only if all 50 passed
# ============================================================

verified_50_df = pd.DataFrame(
    verification_rows
)


assert len(
    verified_50_df
) == 50


print("\nRuns per dataset:")

print(
    verified_50_df.groupby(
        "dataset",
        sort=False,
    ).size()
)


VERIFIED_50_CSV = (
    SUMMARY_DIR
    / "SpaMGCL_all_50_runs_verified.csv"
)


verified_50_df.to_csv(
    VERIFIED_50_CSV,
    index=False,
)


print("\nSaved:")
print(VERIFIED_50_CSV)


print("\n" + "=" * 110)
print("PASS: ALL 50 FORMAL RUNS VERIFIED")
print("=" * 110)

GLOBAL AUDIT: 5 DATASETS × 10 SEEDS × 400 EPOCHS

[HLN-A1]
  PASS seed 0 | ARI=0.220614 | NMI=0.358325
  PASS seed 1 | ARI=0.273804 | NMI=0.393244
  PASS seed 2 | ARI=0.278258 | NMI=0.404178
  PASS seed 3 | ARI=0.237522 | NMI=0.353609
  PASS seed 4 | ARI=0.263325 | NMI=0.387647
  PASS seed 5 | ARI=0.220615 | NMI=0.345039
  PASS seed 6 | ARI=0.296535 | NMI=0.399609
  PASS seed 7 | ARI=0.231655 | NMI=0.342726
  PASS seed 8 | ARI=0.238096 | NMI=0.356490
  PASS seed 9 | ARI=0.232451 | NMI=0.347915

[HLN-D1]
  PASS seed 0 | ARI=0.253712 | NMI=0.338492
  PASS seed 1 | ARI=0.241805 | NMI=0.331744
  PASS seed 2 | ARI=0.268406 | NMI=0.343868
  PASS seed 3 | ARI=0.219876 | NMI=0.300542
  PASS seed 4 | ARI=0.263218 | NMI=0.335151
  PASS seed 5 | ARI=0.240077 | NMI=0.330739
  PASS seed 6 | ARI=0.230640 | NMI=0.320952
  PASS seed 7 | ARI=0.277148 | NMI=0.333270
  PASS seed 8 | ARI=0.241210 | NMI=0.333543
  PASS seed 9 | ARI=0.234592 | NMI=0.341347

[E18.5]
  PASS seed 0 | ARI=0.437536 | NMI=0.55444

Cell 30：官方 audit × 50

In [37]:
# ============================================================
# Cell 30
# Official audit_run.py for all 50 formal runs
# ============================================================

import subprocess
import sys
from pathlib import Path


AUDIT_LOG = (
    SUMMARY_DIR
    / "SpaMGCL_50runs_official_audit.log"
)


audit_log_parts = []
audit_count = 0


print("=" * 100)
print("OFFICIAL SpaMGCL AUDIT: 50 FORMAL RUNS")
print("=" * 100)


for dataset_name, spec in DATASET_SPECS.items():

    slug = spec["slug"]

    print(f"\n[{dataset_name}]")

    for seed in range(10):

        run_dir = (
            FINAL_RESULT_ROOT
            / f"{slug}_formal_400ep_seed{seed}"
        )


        cmd = [
            sys.executable,
            str(
                PROJECT_ROOT
                / "scripts"
                / "audit_run.py"
            ),
            str(run_dir),
        ]


        proc = subprocess.run(
            cmd,
            cwd=PROJECT_ROOT,
            text=True,
            capture_output=True,
        )


        # ----------------------------------------------------
        # Save complete output to log
        # ----------------------------------------------------

        audit_log_parts.append(
            "\n"
            + "=" * 100
            + "\n"
            + f"{dataset_name} | seed {seed}\n"
            + "=" * 100
            + "\n"
            + proc.stdout
            + "\n"
            + proc.stderr
        )


        # ----------------------------------------------------
        # Check return code
        # ----------------------------------------------------

        if proc.returncode != 0:

            print(
                f"  FAIL seed {seed}"
            )

            print(proc.stdout)
            print(proc.stderr)

            raise RuntimeError(
                f"Official audit failed: "
                f"{dataset_name} seed {seed}"
            )


        # ----------------------------------------------------
        # Must contain PASS
        # ----------------------------------------------------

        if "PASS" not in proc.stdout:

            print(
                f"  FAIL seed {seed}: "
                f"PASS not found"
            )

            print(proc.stdout)

            raise RuntimeError(
                f"Audit did not report PASS: "
                f"{dataset_name} seed {seed}"
            )


        audit_count += 1


        print(
            f"  PASS seed {seed}"
            f" | {audit_count:02d}/50"
        )


# ============================================================
# Save complete audit log
# ============================================================

AUDIT_LOG.write_text(
    "\n".join(
        audit_log_parts
    ),
    encoding="utf-8",
)


assert audit_count == 50


print("\n" + "=" * 100)
print("PASS: 50 / 50 OFFICIAL AUDITS")
print("=" * 100)

print("\nAudit log:")
print(AUDIT_LOG)

OFFICIAL SpaMGCL AUDIT: 50 FORMAL RUNS

[HLN-A1]
  PASS seed 0 | 01/50
  PASS seed 1 | 02/50
  PASS seed 2 | 03/50
  PASS seed 3 | 04/50
  PASS seed 4 | 05/50
  PASS seed 5 | 06/50
  PASS seed 6 | 07/50
  PASS seed 7 | 08/50
  PASS seed 8 | 09/50
  PASS seed 9 | 10/50

[HLN-D1]
  PASS seed 0 | 11/50
  PASS seed 1 | 12/50
  PASS seed 2 | 13/50
  PASS seed 3 | 14/50
  PASS seed 4 | 15/50
  PASS seed 5 | 16/50
  PASS seed 6 | 17/50
  PASS seed 7 | 18/50
  PASS seed 8 | 19/50
  PASS seed 9 | 20/50

[E18.5]
  PASS seed 0 | 21/50
  PASS seed 1 | 22/50
  PASS seed 2 | 23/50
  PASS seed 3 | 24/50
  PASS seed 4 | 25/50
  PASS seed 5 | 26/50
  PASS seed 6 | 27/50
  PASS seed 7 | 28/50
  PASS seed 8 | 29/50
  PASS seed 9 | 30/50

[S2-E15]
  PASS seed 0 | 31/50
  PASS seed 1 | 32/50
  PASS seed 2 | 33/50
  PASS seed 3 | 34/50
  PASS seed 4 | 35/50
  PASS seed 5 | 36/50
  PASS seed 6 | 37/50
  PASS seed 7 | 38/50
  PASS seed 8 | 39/50
  PASS seed 9 | 40/50

[S2-E18]
  PASS seed 0 | 41/50
  PASS see

Cell 31：从刚刚验证过的 50 个 run 重建最终总表

In [38]:
# ============================================================
# Cell 31
# Rebuild final paper results from the verified 50 runs
# ============================================================

import numpy as np
import pandas as pd


DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


summary_rows = []


for dataset_name in DATASET_ORDER:

    d = (
        verified_50_df[
            verified_50_df[
                "dataset"
            ] == dataset_name
        ]
        .sort_values("seed")
        .reset_index(drop=True)
    )


    assert len(d) == 10


    summary_rows.append(
        {
            "dataset":
                dataset_name,

            "n_seeds":
                10,

            "epochs":
                400,

            "ARI_mean":
                float(
                    d["ARI"].mean()
                ),

            "ARI_std":
                float(
                    np.std(
                        d["ARI"].values,
                        ddof=0,
                    )
                ),

            "NMI_mean":
                float(
                    d["NMI"].mean()
                ),

            "NMI_std":
                float(
                    np.std(
                        d["NMI"].values,
                        ddof=0,
                    )
                ),

            "raw_ARI_mean":
                float(
                    d["raw_ARI"].mean()
                ),

            "raw_ARI_std":
                float(
                    np.std(
                        d["raw_ARI"].values,
                        ddof=0,
                    )
                ),

            "raw_NMI_mean":
                float(
                    d["raw_NMI"].mean()
                ),

            "raw_NMI_std":
                float(
                    np.std(
                        d["raw_NMI"].values,
                        ddof=0,
                    )
                ),

            "mean_delta_ARI_BSRR":
                float(
                    d[
                        "delta_ARI_BSRR"
                    ].mean()
                ),

            "mean_delta_NMI_BSRR":
                float(
                    d[
                        "delta_NMI_BSRR"
                    ].mean()
                ),

            "BSRR_ARI_improved_seeds":
                int(
                    (
                        d[
                            "delta_ARI_BSRR"
                        ] > 0
                    ).sum()
                ),

            "BSRR_NMI_improved_seeds":
                int(
                    (
                        d[
                            "delta_NMI_BSRR"
                        ] > 0
                    ).sum()
                ),
        }
    )


FINAL_VERIFIED_SUMMARY = pd.DataFrame(
    summary_rows
)


FINAL_VERIFIED_CSV = (
    SUMMARY_DIR
    / (
        "SpaMGCL_5datasets_"
        "10seeds_400ep_"
        "FINAL_VERIFIED.csv"
    )
)


FINAL_VERIFIED_SUMMARY.to_csv(
    FINAL_VERIFIED_CSV,
    index=False,
)


# ============================================================
# Paper-friendly table
# ============================================================

paper_rows = []


for _, row in (
    FINAL_VERIFIED_SUMMARY.iterrows()
):

    paper_rows.append(
        {
            "Dataset":
                row["dataset"],

            "ARI":
                (
                    f"{row['ARI_mean']:.6f}"
                    f" ± "
                    f"{row['ARI_std']:.6f}"
                ),

            "NMI":
                (
                    f"{row['NMI_mean']:.6f}"
                    f" ± "
                    f"{row['NMI_std']:.6f}"
                ),
        }
    )


PAPER_TABLE_DF = pd.DataFrame(
    paper_rows
)


PAPER_TABLE_CSV = (
    SUMMARY_DIR
    / "SpaMGCL_paper_table_results.csv"
)


PAPER_TABLE_DF.to_csv(
    PAPER_TABLE_CSV,
    index=False,
)


# ============================================================
# Macro averages
# ============================================================

macro_ari = float(
    FINAL_VERIFIED_SUMMARY[
        "ARI_mean"
    ].mean()
)

macro_nmi = float(
    FINAL_VERIFIED_SUMMARY[
        "NMI_mean"
    ].mean()
)


macro_delta_ari = float(
    FINAL_VERIFIED_SUMMARY[
        "mean_delta_ARI_BSRR"
    ].mean()
)

macro_delta_nmi = float(
    FINAL_VERIFIED_SUMMARY[
        "mean_delta_NMI_BSRR"
    ].mean()
)


total_ari_improved = int(
    FINAL_VERIFIED_SUMMARY[
        "BSRR_ARI_improved_seeds"
    ].sum()
)

total_nmi_improved = int(
    FINAL_VERIFIED_SUMMARY[
        "BSRR_NMI_improved_seeds"
    ].sum()
)


print("=" * 110)
print("FINAL VERIFIED PAPER RESULTS")
print("=" * 110)

print(
    PAPER_TABLE_DF.to_string(
        index=False
    )
)


print("\nMacro-average over 5 datasets")

print(
    f"ARI = {macro_ari:.6f}"
)

print(
    f"NMI = {macro_nmi:.6f}"
)


print("\nBSRR overall effect")

print(
    f"Macro Delta ARI = "
    f"{macro_delta_ari:+.6f}"
)

print(
    f"Macro Delta NMI = "
    f"{macro_delta_nmi:+.6f}"
)

print(
    f"ARI improved runs = "
    f"{total_ari_improved}/50"
)

print(
    f"NMI improved runs = "
    f"{total_nmi_improved}/50"
)


print("\nSaved:")
print(FINAL_VERIFIED_CSV)
print(PAPER_TABLE_CSV)


print(
    "\nPASS: final paper summary "
    "rebuilt from 50 verified runs."
)

FINAL VERIFIED PAPER RESULTS
Dataset                 ARI                 NMI
 HLN-A1 0.249288 ± 0.025243 0.368878 ± 0.023078
 HLN-D1 0.247068 ± 0.017149 0.330965 ± 0.011777
  E18.5 0.427532 ± 0.064947 0.558228 ± 0.016322
 S2-E15 0.381765 ± 0.032992 0.568241 ± 0.014581
 S2-E18 0.358493 ± 0.036774 0.502234 ± 0.016871

Macro-average over 5 datasets
ARI = 0.332829
NMI = 0.465709

BSRR overall effect
Macro Delta ARI = +0.014273
Macro Delta NMI = +0.005303
ARI improved runs = 32/50
NMI improved runs = 31/50

Saved:
/kaggle/working/SpaMGCL/SpaMGCL/formal_summary_400/SpaMGCL_5datasets_10seeds_400ep_FINAL_VERIFIED.csv
/kaggle/working/SpaMGCL/SpaMGCL/formal_summary_400/SpaMGCL_paper_table_results.csv

PASS: final paper summary rebuilt from 50 verified runs.


Cell 32：生成最终协议 Manifest

In [39]:
# ============================================================
# Cell 32
# Final formal experiment protocol manifest
# ============================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import platform
import subprocess

import numpy as np
import sklearn
import torch


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ============================================================
# Git commit
# ============================================================

git_commit = (
    subprocess.check_output(
        [
            "git",
            "-C",
            str(PROJECT_ROOT.parent),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    )
    .strip()
)


# ============================================================
# Critical source hashes
# ============================================================

critical_source_files = [
    RUNNER_FORMAL,

    PROJECT_ROOT
    / "scripts"
    / "audit_run.py",

    PROJECT_ROOT
    / "src"
    / "clustering"
    / "refinement.py",

    PROJECT_ROOT
    / "src"
    / "clustering"
    / "predict.py",
]


source_hashes = {}


for path in critical_source_files:

    assert path.exists()

    source_hashes[
        str(
            path.relative_to(
                PROJECT_ROOT
            )
        )
    ] = sha256_file(path)


# ============================================================
# Per-dataset archives
# ============================================================

dataset_archive_names = [
    "SpaMGCL_HLNA1_10seeds_400ep_FINAL.zip",
    "SpaMGCL_HLND1_10seeds_400ep_FINAL.zip",
    "SpaMGCL_E185_10seeds_400ep_FINAL.zip",
    "SpaMGCL_S2E15_10seeds_400ep_FINAL.zip",
    "SpaMGCL_S2E18_10seeds_400ep_FINAL.zip",
]


dataset_archives = {}


for filename in dataset_archive_names:

    path = (
        Path("/kaggle/working")
        / filename
    )


    if path.exists():

        dataset_archives[
            filename
        ] = {
            "exists":
                True,

            "size_bytes":
                path.stat().st_size,

            "sha256":
                sha256_file(path),
        }

    else:

        dataset_archives[
            filename
        ] = {
            "exists":
                False,
        }


gpu_name = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else None
)


FINAL_PROTOCOL_MANIFEST = {

    "experiment_name":
        (
            "SpaMGCL Formal Benchmark "
            "5 Datasets x 10 Seeds x 400 Epochs"
        ),

    "created_at":
        datetime.now().isoformat(),

    "git_commit":
        git_commit,

    "datasets":
        DATASET_ORDER,

    "training": {
        "epochs":
            400,

        "training_seeds":
            list(range(10)),

        "n_runs_per_dataset":
            10,

        "total_runs":
            50,

        "warm_up_epochs":
            10,

        "fresh_training":
            True,

        "sensitivity_checkpoints_reused":
            False,
    },

    "official_readout": {
        "embedding":
            "concat_z",

        "refinement":
            "bsrr",

        "bsrr_spatial_k":
            3,

        "clustering":
            "kmeans",

        "kmeans_n_init":
            20,

        "kmeans_random_state":
            0,
    },

    "evaluation": {
        "ARI":
            "sklearn adjusted_rand_score",

        "NMI":
            (
                "sklearn "
                "normalized_mutual_info_score"
            ),

        "nmi_average_method":
            "max",

        "summary_std_ddof":
            0,
    },

    "environment": {
        "python":
            platform.python_version(),

        "torch":
            torch.__version__,

        "cuda_runtime":
            torch.version.cuda,

        "gpu":
            gpu_name,

        "numpy":
            np.__version__,

        "sklearn":
            sklearn.__version__,
    },

    "source_sha256":
        source_hashes,

    "dataset_archives":
        dataset_archives,

    "formal_summary":
        (
            FINAL_VERIFIED_SUMMARY
            .to_dict(
                orient="records"
            )
        ),
}


PROTOCOL_MANIFEST_PATH = (
    SUMMARY_DIR
    / "SpaMGCL_FINAL_PROTOCOL_MANIFEST.json"
)


PROTOCOL_MANIFEST_PATH.write_text(
    json.dumps(
        FINAL_PROTOCOL_MANIFEST,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 100)
print("FINAL PROTOCOL MANIFEST")
print("=" * 100)

print(
    "Git commit:",
    git_commit,
)

print(
    "GPU:",
    gpu_name,
)

print(
    "Total formal runs:",
    50,
)


print("\nSource hashes:")

for key, value in (
    source_hashes.items()
):

    print(
        f"{key:45s} "
        f"{value[:20]}..."
    )


print("\nPer-dataset archives:")

for filename, info in (
    dataset_archives.items()
):

    print(
        f"{'OK' if info['exists'] else 'MISSING':7s}"
        f" {filename}"
    )


print("\nSaved:")
print(PROTOCOL_MANIFEST_PATH)


print(
    "\nPASS: final protocol "
    "manifest created."
)

FINAL PROTOCOL MANIFEST
Git commit: b4abd3de0c590e024a28add6fba4f0f089837112
GPU: Tesla T4
Total formal runs: 50

Source hashes:
experiments/run_exp_formal_400.py             d2ec5980ff99fb596dfd...
scripts/audit_run.py                          f985d0099d96bfbbe74a...
src/clustering/refinement.py                  4b0f0331129e057d34c7...
src/clustering/predict.py                     abc17d7ee42e618b90c9...

Per-dataset archives:
OK      SpaMGCL_HLNA1_10seeds_400ep_FINAL.zip
OK      SpaMGCL_HLND1_10seeds_400ep_FINAL.zip
OK      SpaMGCL_E185_10seeds_400ep_FINAL.zip
OK      SpaMGCL_S2E15_10seeds_400ep_FINAL.zip
OK      SpaMGCL_S2E18_10seeds_400ep_FINAL.zip

Saved:
/kaggle/working/SpaMGCL/SpaMGCL/formal_summary_400/SpaMGCL_FINAL_PROTOCOL_MANIFEST.json

PASS: final protocol manifest created.


Cell 33：制作主实验 MASTER ZIP

In [40]:
# ============================================================
# Cell 33
# Final 5-dataset master archive
# ============================================================

from pathlib import Path
import zipfile


MASTER_ARCHIVE = Path(
    "/kaggle/working/"
    "SpaMGCL_5Datasets_10Seeds_400ep_MASTER.zip"
)


MASTER_CHECKSUM_FILE = (
    SUMMARY_DIR
    / "SpaMGCL_MASTER_SHA256SUMS.txt"
)


MASTER_RUN_FILES = [
    "metrics.json",
    "config.yaml",
    "manifest.json",

    "gt_labels.npy",
    "coords.npy",

    "pred_labels.npy",
    "pred_concat_z_kmeans.npy",
    "pred_q_argmax.npy",

    "z_concat.npy",
    "z_bsrr.npy",

    "metrics_epoch400.json",
]


archive_entries = []


# ============================================================
# 50 formal runs
# ============================================================

for dataset_name, spec in (
    DATASET_SPECS.items()
):

    slug = spec["slug"]

    for seed in range(10):

        run_dir = (
            FINAL_RESULT_ROOT
            / (
                f"{slug}_"
                f"formal_400ep_"
                f"seed{seed}"
            )
        )


        for filename in (
            MASTER_RUN_FILES
        ):

            file_path = (
                run_dir
                / filename
            )


            assert file_path.exists(), (
                f"Missing master artifact: "
                f"{file_path}"
            )


            arcname = (
                Path(
                    "results_final_400"
                )
                / run_dir.name
                / filename
            )


            archive_entries.append(
                (
                    file_path,
                    arcname,
                )
            )


# ============================================================
# 50 original YAML configs
# ============================================================

for dataset_name, spec in (
    DATASET_SPECS.items()
):

    slug = spec["slug"]

    for seed in range(10):

        cfg_path = (
            FINAL_CONFIG_DIR
            / (
                f"{slug}_"
                f"formal_400ep_"
                f"seed{seed}.yaml"
            )
        )


        assert cfg_path.exists()


        archive_entries.append(
            (
                cfg_path,
                Path(
                    "configs/formal_400ep"
                )
                / cfg_path.name,
            )
        )


# ============================================================
# Critical source
# ============================================================

source_entries = [
    (
        RUNNER_FORMAL,
        Path(
            "experiments/"
            "run_exp_formal_400.py"
        ),
    ),

    (
        PROJECT_ROOT
        / "scripts"
        / "audit_run.py",

        Path(
            "scripts/audit_run.py"
        ),
    ),

    (
        PROJECT_ROOT
        / "src"
        / "clustering"
        / "refinement.py",

        Path(
            "src/clustering/"
            "refinement.py"
        ),
    ),

    (
        PROJECT_ROOT
        / "src"
        / "clustering"
        / "predict.py",

        Path(
            "src/clustering/"
            "predict.py"
        ),
    ),
]


archive_entries.extend(
    source_entries
)


# ============================================================
# Summary / audit files
# ============================================================

summary_files_for_master = [
    VERIFIED_50_CSV,
    FINAL_VERIFIED_CSV,
    PAPER_TABLE_CSV,
    AUDIT_LOG,
    PROTOCOL_MANIFEST_PATH,
]


for slug in [
    "hlna1",
    "d1",
    "e185",
    "s2e15",
    "s2e18",
]:

    summary_files_for_master.extend(
        [
            SUMMARY_DIR
            / (
                f"{slug}_"
                "10seed_400ep_raw.csv"
            ),

            SUMMARY_DIR
            / (
                f"{slug}_"
                "10seed_400ep_summary.csv"
            ),
        ]
    )


for file_path in (
    summary_files_for_master
):

    assert file_path.exists(), (
        f"Missing summary file: "
        f"{file_path}"
    )


    archive_entries.append(
        (
            file_path,

            Path(
                "formal_summary_400"
            )
            / file_path.name,
        )
    )


# ============================================================
# Duplicate check
# ============================================================

archive_names = [
    str(arcname)
    for _, arcname
    in archive_entries
]


assert (
    len(archive_names)
    ==
    len(set(archive_names))
), "Duplicate archive paths detected."


# ============================================================
# SHA256 checksums
# ============================================================

checksum_lines = []


for file_path, arcname in (
    archive_entries
):

    checksum_lines.append(
        (
            f"{sha256_file(file_path)}"
            f"  "
            f"{arcname}"
        )
    )


MASTER_CHECKSUM_FILE.write_text(
    "\n".join(
        checksum_lines
    )
    + "\n",
    encoding="utf-8",
)


archive_entries.append(
    (
        MASTER_CHECKSUM_FILE,

        Path(
            "formal_summary_400"
        )
        / MASTER_CHECKSUM_FILE.name,
    )
)


# ============================================================
# Create ZIP
# ============================================================

if MASTER_ARCHIVE.exists():

    MASTER_ARCHIVE.unlink()


with zipfile.ZipFile(
    MASTER_ARCHIVE,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    for file_path, arcname in (
        archive_entries
    ):

        zf.write(
            file_path,
            arcname=str(
                arcname
            ),
        )


# ============================================================
# ZIP integrity test
# ============================================================

with zipfile.ZipFile(
    MASTER_ARCHIVE,
    mode="r",
) as zf:

    bad_file = zf.testzip()

    names = zf.namelist()


assert bad_file is None

assert len(names) > 0


size_mb = (
    MASTER_ARCHIVE.stat().st_size
    / 1024
    / 1024
)


master_sha256 = sha256_file(
    MASTER_ARCHIVE
)


print("=" * 100)
print("SpaMGCL FINAL MASTER ARCHIVE")
print("=" * 100)

print(
    "Archive:",
    MASTER_ARCHIVE,
)

print(
    "Files:",
    len(names),
)

print(
    f"Size: {size_mb:.2f} MB"
)

print(
    "SHA256:",
    master_sha256,
)


print("\nZIP integrity: PASS")


print(
    "\nRecovery checkpoints "
    "100/200/300 are excluded."
)

print(
    "Final model checkpoints "
    "remain in each dataset FINAL ZIP."
)


print(
    "\nPASS: SpaMGCL formal "
    "main experiment is frozen."
)

SpaMGCL FINAL MASTER ARCHIVE
Archive: /kaggle/working/SpaMGCL_5Datasets_10Seeds_400ep_MASTER.zip
Files: 620
Size: 297.39 MB
SHA256: ed5ecb7a3e162965970202cc7c58cb952538344bbd95bb929076477277db7b63

ZIP integrity: PASS

Recovery checkpoints 100/200/300 are excluded.
Final model checkpoints remain in each dataset FINAL ZIP.

PASS: SpaMGCL formal main experiment is frozen.
